<a href="https://colab.research.google.com/github/Trixel3/Xdocuments/blob/main/Copie_de_Copie_de_Objectif_IA_Atelier_Code_03_09_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Votre propre assistant IA, construit et mis en ligne ce soir

**Préparé par Machine Learnia (https://www.machinelearnia.com/)**

Bienvenue ! Dans ce notebook, nous allons construire un assistant IA capable de répondre à des questions sur un ensemble de documents que vous allez **aller chercher vous-même sur Internet**, puis nous le  mettrons **en ligne**, à une adresse **éphémaire** que vous pourrez ensuite envoyer à qui vous voulez.

(Cette adresse sera unique pour chacun d'entres vous)

**Si vous débutez, voila ce qu'il faut savoir**

- Vous exécutez chaque cellule avec **Shift + Entrée**, en même temps que moi.
- Certaines cellules ont des **trous** `______` : on les remplit ensemble, en direct. Ce sont les lignes qui comptent. `LES RÉPONSES SONT DONNÉES A LA FIN DU NOTEBOOK`
- Vous n'avez pas besoin de tout comprendre ce soir. Vous avez besoin de tout **faire**. Les explications arrivent au fur et à mesure.
- Une erreur ? Vérifiez que vous avez bien exécuté les cellules précédentes dans l'ordre, ou demandez dans le chat.

*Première étape : menu **« Fichier → Enregistrer une copie dans Drive »** pour travailler sur VOTRE copie.*

## Jalon 0 · Préparation (~2 minutes)

**Avant de lancer** : menu « Exécution » → « Modifier le type d'exécution » → choisissez **« T4 GPU »** s'il est proposé. Pas de GPU disponible ? Tout marchera quand même, un peu plus lentement.

On installe les outils du soir : le lecteur de PDF, le moteur de recherche sémantique, et l'atelier de mise en ligne.

In [1]:
%pip install -q sentence-transformers gradio pypdf
print("✅ Installation terminée !")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 18.8 MB/s eta 0:00:00
✅ Installation terminée !


In [2]:
import torch

if torch.cuda.is_available():
    MODELE = "Qwen/Qwen2.5-1.5B-Instruct"
    DEVICE = 0
    print("🚀 GPU détecté : on utilise le moteur rapide.")
else:
    MODELE = "Qwen/Qwen2.5-0.5B-Instruct"
    DEVICE = -1
    print("🐢 Pas de GPU : on utilise le moteur léger. Tout marchera quand même.")

print()
print("👉 Écrivez PRÊT dans le chat !")

🐢 Pas de GPU : on utilise le moteur léger. Tout marchera quand même.

👉 Écrivez PRÊT dans le chat !


In [3]:
#@title 🛟 Documents de secours (cellule repliée : exécutez-la, ne l'ouvrez pas) { display-mode: "form" }
# Les six PDF de l'atelier, encodés en base64. Ils ne servent QUE si le téléchargement
# de la cellule suivante échoue. Un système en production a toujours un chemin de repli.
PDF_SECOURS = {
    "club-odyssee-conditions-generales.pdf": "JVBERi0xLjQKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUgo+PgplbmRvYmoKMiAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYSAvRW5jb2RpbmcgL1dpbkFuc2lFbmNvZGluZyAvTmFtZSAvRjEgL1N1YnR5cGUgL1R5cGUxIC9UeXBlIC9Gb250Cj4+CmVuZG9iagozIDAgb2JqCjw8Ci9CYXNlRm9udCAvSGVsdmV0aWNhLUJvbGQgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YyIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNCAwIG9iago8PAovQ29udGVudHMgMTAgMCBSIC9NZWRpYUJveCBbIDAgMCA1OTUuMjc1NiA4NDEuODg5OCBdIC9QYXJlbnQgOSAwIFIgL1Jlc291cmNlcyA8PAovRm9udCAxIDAgUiAvUHJvY1NldCBbIC9QREYgL1RleHQgL0ltYWdlQiAvSW1hZ2VDIC9JbWFnZUkgXQo+PiAvUm90YXRlIDAgL1RyYW5zIDw8Cgo+PiAKICAvVHlwZSAvUGFnZQo+PgplbmRvYmoKNSAwIG9iago8PAovQ29udGVudHMgMTEgMCBSIC9NZWRpYUJveCBbIDAgMCA1OTUuMjc1NiA4NDEuODg5OCBdIC9QYXJlbnQgOSAwIFIgL1Jlc291cmNlcyA8PAovRm9udCAxIDAgUiAvUHJvY1NldCBbIC9QREYgL1RleHQgL0ltYWdlQiAvSW1hZ2VDIC9JbWFnZUkgXQo+PiAvUm90YXRlIDAgL1RyYW5zIDw8Cgo+PiAKICAvVHlwZSAvUGFnZQo+PgplbmRvYmoKNiAwIG9iago8PAovQ29udGVudHMgMTIgMCBSIC9NZWRpYUJveCBbIDAgMCA1OTUuMjc1NiA4NDEuODg5OCBdIC9QYXJlbnQgOSAwIFIgL1Jlc291cmNlcyA8PAovRm9udCAxIDAgUiAvUHJvY1NldCBbIC9QREYgL1RleHQgL0ltYWdlQiAvSW1hZ2VDIC9JbWFnZUkgXQo+PiAvUm90YXRlIDAgL1RyYW5zIDw8Cgo+PiAKICAvVHlwZSAvUGFnZQo+PgplbmRvYmoKNyAwIG9iago8PAovUGFnZU1vZGUgL1VzZU5vbmUgL1BhZ2VzIDkgMCBSIC9UeXBlIC9DYXRhbG9nCj4+CmVuZG9iago4IDAgb2JqCjw8Ci9BdXRob3IgKENsdWIgT2R5c3NcMzUxZSBcKGZyYW5jaGlzZSBmaWN0aXZlXCkpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA5MDIxNzMyMDgrMDEnMDAnKSAvQ3JlYXRvciAoXCh1bnNwZWNpZmllZFwpKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA5MDIxNzMyMDgrMDEnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAoXCh1bnNwZWNpZmllZFwpKSAvVGl0bGUgKENsdWIgT2R5c3NcMzUxZSAtIENvbmRpdGlvbnMgZ1wzNTFuXDM1MXJhbGVzIGQnYWJvbm5lbWVudCkgL1RyYXBwZWQgL0ZhbHNlCj4+CmVuZG9iago5IDAgb2JqCjw8Ci9Db3VudCAzIC9LaWRzIFsgNCAwIFIgNSAwIFIgNiAwIFIgXSAvVHlwZSAvUGFnZXMKPj4KZW5kb2JqCjEwIDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoIDIxMjQKPj4Kc3RyZWFtCkdhdG08Z04pJSwmOk1sK2xxOWlkN29pNEhTaWU9JTFYaEBHOE5VW1pxdVciQiRrSTdXI1c7R1RTbUAvSixnMkxPa3I5cDIrR2RfJUYqPFw1LDVmQ01SMi8xNDxxUSIoUixXZCIpaT8jPCQqRVhaRzRrIkBScztcaFFBZkchQS9XakQ0PDtEYWZiM2M0RT9wVjRwJ2pjUV0wZDdFP2Y4a1VQKSluLSZqQzRwKzNpcjNgPCR1WiktdShaSVNISD9EcEkxIlZtMys3XHVFaXIxbztgQ0QwYDhFJiQnYTxJXGJMPVhhWyEoPilwVl9yO2tTOlhmSmJXS2VBajxfZlwpMk8ucDxuJSFyUSYrOEBpNDdRYG1AIXMrMSxWaWNwLmRoIyxGJGlbYCIqOFQjbXFtdCdJV24pLUxtYDFOSW5wckVhMCFJU1llbCJwIW47VkhXKWxxLVJfKU9UUF5VODFpTzw2R2YycVclK1AqRjBbRCQyTiUqOyhxVmgoWT5FMDokNSg8W0prXDFsRHApRCtrO19BJUdISkY4RTdtZUJNUVtWWHM9SEhdKjBBXylVL0g6dUhKPUxHW1BJQ1s9a2BjSW9Qcy1lWXBjTmN1KmhvVkhjbS5kcDhObjFAUmBwS1QjdUAnMzZTNkRsNl0tJTRqMFJibEoqQVc8R3FhdShObG9vU1VVVGxPN09ZKCYxWyYqXDpMU2BXSDdmckQrbGRdMmp1SSpuTG1TI1dGR2RlL1VSQ208SXFERmNeLi1oI2k6S1o7bj1GSVI+QkRQLmhHP1RjQFdRdGUrc0gvWE9KYXFnU2BdVlNpU0skakIqRycuNUAxJGY+ImF1VD0jbyFbPi9kdVotIiZQN0ttUjJhUHVLN1h0NCVGb291KVtTV1xbbE5wOlskIShWbE9FRGI+TGZyWlBqcE9USERjKSs+cmhdUDpnJUEyRyFrOSgpUGg2MWZYKTw0OV1JZV9lR0VHIltdMXBrRSFxImIoa1QlUGFJYyY3b0t1JURgMy9SMSJjaWZyNDY3JW82QyVKVTAhTyUsTTpsPUxyI0ZTSSxCSmxNJWN1TCtRLU0vMURZLEddM2g5TSZgUUtOIyFYWD8uKTpVUEBHTTI3bF1fakIzJVNVZE1BWVk1PzlyaSVaOWcpbUUxJ0dCbFN0Jz5eUzglVHJHdCxgOiMuTUM6bS5lTCJEZFtCTV0rM1lcQVg/WmhEPDl0JVJATXJoJUZvJ2MvXTREOktyMmNaUGMoNjJtUlRmO29HOFgoXCcsJm5nZWJnNkpMWnJWW14rNj1QbE9RZFMhJ0RkR3AtKTk2ZEYzcyViY0wrR1NkNUxaLjJyN2IqRSZWZjNrUlNWNkhdPWVZLzI3YShHLCdNW3EkZFMsbEooaUxNQShVdFBnbDZWXU0+UltpUWJbYWBNZUdpU1MvUDJSdG5LNzEkbzFJaj0lIyQ+YjAqKWAtPiImI0R1UzwrZixbR2dJPmYyLmxvcCNGc0FKaSlfX3ItZUtiJUQ8PWI7QUFYMHFWX00pIXIhQkRYSjpsPDNga1BYPVRiWXM8Ryo/V0pGSSUuMVNcJV9ldCgwX0M1MCFVRUM2TkRwcFtvLnE9R2Y8P01AW29pcy1xWVJHVC8yZm1tImM2RjY/WWg3X1RoLnFKbk85Ym5vb0wtZidMPGRvaCI9QSIzRldfZiY2Lik3Kz5iZixuYD9sVG09aidNPTFSaUtKODohVWQ0ZC9QNWo/JGk5KGYqTSQzJDlCTFM5JispLCp1XzwuNl0tRjtqPlNrcC1QL2YnKzYndTs/JSRsKWsiX2paSkkvWHNtZmwsKFB1cDIsIlFVY1w+ZF5xKmI2MEZVRENEUj5lR3J0JjF0NCcxZnMrZWEhTl4mPk9ZZmIqXXBETFdOJS85OTYqRlw5cWxgaGNTSSQsN2QkNSZcX2JGW2dqZUhcP2RcKVheUFdQQmAhM2UuUyJTRC9waXNyZyxtL1VoaFZiTW5ZLVhnXyFbYmYuQikvPmFkYExiZ3VocDBtXFNYQlAiSTphI3InbyVRZDI9QlEpSVFXWltSNEQtUE9HSC10PDJVNzE8ay8rN1gvP1xZIl9FazlwSlBUOS4uYi5eIk41I3VwIU4uI11fUWdKczNbYHVsajdzL15FXTgkOGlxRVRSQm4iKFxyIWJjMjduRTtMdDltPDRCTlV0SCtNJC5UJTlqIW9GZGdoJ2Y0Uys2KGtpOjwmQktQYV9aZWZzKjpfMXBsXVZHSGRrJiU1XnMkaV1Cc2BAVk4mVk5PckFlZGFEVylUZkolWGE4Jy0tKj9BKDlYY1koTkA6KlJtZlNKXEUqUi00NjNtKSs1UlQ7JkBsNGs7KztFWmFYUU9bRitFaio1VjBZPEpLak1xTj9kJ0lPSDJYaEBFWWUxLT40U0tyPSRHQD5JRzBDYWs/XWZFSCcqY3BqZzxsdUgkSCdHImVnUnAiaF9bO2JZR0suT0o5OlBWOU9cXSVlInFlRUlEZWl0P0U8MERIcXM4Wz8ySmldSz8iZlwxQDlAWElDLSVjc0daRipCLE0tb0RRJm91YGAjPz04JFo1UDI8ZU5JN0ZELmAvTXRpQig1dFg5QywyRCRYSk5KST9ialFuZ0wqUUY7ZSZEXjBiPl1JXT9EcEAmPWFdPGtdTipWNVdFSlYxLldhPjgiLjc7WXI9JFQ1X2M/ImdmcmBLKz4sS3VLY3M4b2QhalFtR28rTEIiNThbSFxyNSRkXTEzQjBRT2UnVWNkbEAnYzhCLCZgInBaYU0yPy1sKCJeYXQnNDNWOjtBcSxRamc+LjRQNCM2dS9HWEhFSyJpbUM3Im9+PmVuZHN0cmVhbQplbmRvYmoKMTEgMCBvYmoKPDwKL0ZpbHRlciBbIC9BU0NJSTg1RGVjb2RlIC9GbGF0ZURlY29kZSBdIC9MZW5ndGggMjMwNwo+PgpzdHJlYW0KR2F0VTRiQXVUMyddJSk6b1pqJEI2UCgpRWIqdSg+LTdgLy9QKXU0bk5SWm0wOE0uSXBgOk5eYHBOR2w9X2hLWS1YIllbXDUwbDdgMk9UcmowQjpacycmcllOcSI5XEFBNWZiZU0hU2AkLD8+PzdxcylZUlZfJFlMNCEhMzJWYj1bKmNIIlcvRyk+QyRILD5wOHAoSnVmU15HRG00O1klSmc3Nk46WjpbJFkrL0gtJVpbKFx0aCs9ZWMsNzxPIz8zYS0sQVo5XlIyYmw9WTJIIm5cSmVkQzRDbW44Ym5iLDIqVD9gUSYyJWhbI1ZANjVwYTRdaj4oQ1RjV0cqTUkhODZeSFdYKFNeNjc5aVgtUDZyKW5qQSRVSDhnT2ciWmpwQ1YraVoyTyVwLjJIOmUjKj9IKk9kKiNPUmUkJWRrV0o0aklwbGlvXDxbSE1WRlA7USxjSCkwS2lfK0khSktGSj86LkNTc2svS1JuLjpQOzNHNiQqMV1vIUM4LmgoR0syWHA8VWpxIk1RVkE2XSRQNTguU0w+V2AiUVlWKCVYS2tCMT9PTlZLNTRHTWolKzhFN2NoTCFTM1RzNjREPFo0Xk8iKURZR2Y9RVRUWVkmMlg6a0ZCPUZmXllOWUNgWVlsKEI1U047RUloZTNJQmhAR0ViZDJsX11eMCcyP2Bub1xIXj47TCdZWGFgQ0QnUzRwJEs/NWshRUMzWFtpM2BgJEkudGIjVWA6T1tPMGBWMyhBbUg/Q3BbTj0lIy1LZTJscS5ELnMrUzNkKHImNHBuSEE/XGFDWWtMNCwzSzJsWko1bjAjWmZCRnM0U1IyS1lSS1ZiPjpSNCEyRT1ETCJGJWpdYi5NMSlXayckXW5BaXJXREhsV2AwTTNOdF5Da0AsXExDLkJqbXA8Qj0qcmVWIzBhS2VjbENOTVFhcE1qK0BTK1YxbDc0ZnQ6KENFdVdZRHRCRiU/WEpGaTxEQWRwZXQncWFvRThMMiI1bVI4MkcndV1ATSZtWVo/MmQ2Yz5xR1xTWTMubjxDcCw+MG9gT0FtcmMyKHApOWBDS0RSNVJfdG0jI0hEL2hqXWg2Ly4hI3QwIl9PVDBHNyotaD5eZUM9KSJkOjI6R1NrWFUmTHRhYzA5KSJPNlZQZm5mPlsnOFBxZWhecU8iO2MkRFdFcUUsM1BoMG9iR2M+YUJaci5FZSpcK1tENGQoQ1BwMjRdazNYZVBmai1NXyxAaGNxSW4qN3VkRFFCPE5hSy41b21WRWU+Ll9hKHFjMClcP11kKz02XGg1XHQhISw2NkwmYmFIZDEhOkBEQlxWbl0tVmMwMlk6MmZRNCsoMzdhX2M5SkE4Nlk0LSkhLWhENzhjUSs3Uz88LVpOQjBjIjpAX11SajxwM1ZoJyFwT1FYY0xpKl41WGltVWEhWSQhP08jVmA1OjFLTkYyMmYyXz1AbTYzaVUmZnFaMnVGZT8lXGIiRUZzLDMhI0VbYWosQjpEPGYtb3I1NXRvbVlfVT9yZU9STm8uViEwKF9lIkRsLVxkSHNuVDcvOCJCW0NuS2kpITVcZFknZmRhTlxVSSstN3MsTyVJTyQpZFxWMjQxM05jImskdStbMltEbyVTczxxNy1NXC1kK0tAOlVaZGFVamlAVjNRdV8lMz5LKCM5YjxiIixkI2BYO1hENSlYJmVaZjonZllebFlrS2QuUVAmXk9VdWopTl06OkdnKytqQysvNz1kQUBdMTs3UjIxRThbMzBzXl5gKCtBS3JaQilLUStuSWlAVi9NTXNuPUklIkpANUg8Ki4mTmxqT3BDJFBvb0RDQFtOaUgvZTZUNSY4YWRzYWFPIzZJJyRgQm40RkB0J0hXMlxmWS81ZFlZTihBJ2BZaEtjSS8qVSNLbVRuLUktR15qXlU8UkZEUlBULm1rJm8iJUdvcDMsLXAjQ0liZ2QjRm9qSWhdT25jQ1wvTSZ0JVdcUnFmOlpTWGFDcyxCVDJXJkUpYSVjLVRVNXFIbFlYS0NmcEpsUGo9aFUyUnBUQTxgUF5pcUxEKkZbPXBYXiRKX2E+SDxTL05yRFdHR1w6akc8VmJjci1VcCQqKG1ybSZoNHAlT2A4cFMiR0s8cTozNW11VWNHUWtYQixLXS1TYVVRNkBWTFcmYDh1dGw3bXM0OEE5NnItMy9xN00oaj5wNnErLTBnOC03NUxSdCVoaWtwVlVZMUFwJzhkK0JlXyhbTiJeZFcjVm1nIldlOldGMDpYQkRgYFk4Q2ZXYD8wNVZOdGNFOyRfQGtESzw2T2toZUAiJlc8NlxSKmgxZGw0c0peMz0sMzNkQmNiPF1sXmFuTz5HP1JuSkVUbzcmW0FpRGZnNiUmTlQobU4pREI4aCpGLyNkL3NDVms6QUssZkc5Ik4/al06K3EqVydOVy5HdTNkYHRbbk8qZE9tckwtcDxSa3NNQ2Q7ZDY4I1c8X1FZYD5fL1ItLG1XREIiPEc0ZSoyVzpRX2JvdVdrS0RERV9zWl82Y2BqXWluKz5MVnQ2SilxV1VyNCdbUytpNS1GZERmM0kiXVJoPihsRD8nQkVCUE5QKzgyTGI0cDA2LVY5bWNmbm9udG1BVihcVW4uUklvcEM1Yjw/QFM7KD44WmIzM0k5Nk1oQzhrU2hyQmBnOGdpIlJSIStSUVhWJC9HZidTayhGN29QPltkSzRMMTM5WCtBJTInLkhnYE5yUzs2U2VRTGRpcyooZixFNTlATlVdVER0W2lgJVhRNnA5Kl9aUSJ0OkclTTFsSUltR2RZOCEvRVlfTS0nTU9AY1AsQHEiN0FNU1xrJTNbPWowLihnZVk4LUJfSThnSjtYSE51VExZWXNvTStnMC5TOm8sRHFNRz1GbUUtQCJyWl0mWHVCTjhiV2NEW0wtU0xkRVwlX1tDT2cjVSVdUz5RKXUyLUVzV1FkMzNMbkAhVSRKJ1c4R0dhIWZdLi9cRXVwUSVHQlRSSzouUzNAWSVIMVt1TzEoSzNpNCNULFxHQ3IyQUtRNDs3I05waHBwJ0QuYCM5K1VQJFtVPiUkcyRUSXBVYTNOUFBGYzchJiVUJH4+ZW5kc3RyZWFtCmVuZG9iagoxMiAwIG9iago8PAovRmlsdGVyIFsgL0FTQ0lJODVEZWNvZGUgL0ZsYXRlRGVjb2RlIF0gL0xlbmd0aCA2NTYKPj4Kc3RyZWFtCkdhc2FsYkFRJmcmQTdsampxYkVTPCleNmRrYTJLJTpyI0BuNWVaOEc5OWAoOiwlV3JrcEAjVmhqQDk3Zi9ncSFRLzgnLT1RNCstPi9IMltbR1YmQ1gpWnJgMTJJTlJyS2xWaC9FZCFoVC41RlxfXUxXRWJGaUclcGo5PUE3aiRjJ2ddLDtjcicuQCc0KTlRajg+X0lBZiVARDQ7XWBsVGBmQT4tX2NPKERyVyEySXFBXT86KS5XJnFtX0g6TG09Ny1JTTQyMDY7S2smajIlW0thPCU8ZElmMDpoPFtFYkdXZiY1YypnS0pycTlqZSNCOzsxP050ZGBValhqJWE/Qz4vdGxHOGVdYS45Wm9vbl1FcV0rbUx1MDxDcVM+S1hHbTgmJldMPnM2MCJDamwidCxdPnVJbjw8OTY8RG5tSXVoTG0nYVNkWDcjaj8tPl08O28xN2o8UWhWUFwnckYpaWFzKzBtJTRjKUxvclxhJTh1NCxHTDFnRiclKEZZJHUkZFMqIUsjNENAQDEwO3NxTGBdJi5YOmFiLFhvaTtQWjFOaDRDXXBZOCNVPmRiPjMobTxPYSQ1M2UybjZlbj47cEw9TTY9RCJTYVFpc25lSlNvJzA0QDkvWGJcO1M5YzpEc101MzRbRVpFKHNBTGlKLmVsJ1FKcS9CZVw2VXRGW2hKYG1zPSg/LWdtQlBybDkmPytgWGx1YGVcIm0zPThGdWpwVmdOZGBuVHIhX0lcVENwSU5RNzBTdW9gIVM1LGouOlRgTVxGRWdMNi50UF1aPFcvKVYrU2s2XitBZyYqYS9FPSJwTWlebC81I1dJaCRYUiluR09OQ1xlZSsjJCJHM2k7M0hEWyc9NUdQRyJMPH4+ZW5kc3RyZWFtCmVuZG9iagp4cmVmCjAgMTMKMDAwMDAwMDAwMCA2NTUzNSBmIAowMDAwMDAwMDYxIDAwMDAwIG4gCjAwMDAwMDAxMDIgMDAwMDAgbiAKMDAwMDAwMDIwOSAwMDAwMCBuIAowMDAwMDAwMzIxIDAwMDAwIG4gCjAwMDAwMDA1MjUgMDAwMDAgbiAKMDAwMDAwMDcyOSAwMDAwMCBuIAowMDAwMDAwOTMzIDAwMDAwIG4gCjAwMDAwMDEwMDEgMDAwMDAgbiAKMDAwMDAwMTM0OSAwMDAwMCBuIAowMDAwMDAxNDIwIDAwMDAwIG4gCjAwMDAwMDM2MzYgMDAwMDAgbiAKMDAwMDAwNjAzNSAwMDAwMCBuIAp0cmFpbGVyCjw8Ci9JRCAKWzwwOThhOTdmZDZmM2ZlZWY3MGE2Y2Q1Yjg1MWU1ODE2ND48MDk4YTk3ZmQ2ZjNmZWVmNzBhNmNkNWI4NTFlNTgxNjQ+XQolIFJlcG9ydExhYiBnZW5lcmF0ZWQgUERGIGRvY3VtZW50IC0tIGRpZ2VzdCAob3BlbnNvdXJjZSkKCi9JbmZvIDggMCBSCi9Sb290IDcgMCBSCi9TaXplIDEzCj4+CnN0YXJ0eHJlZgo2NzgyCiUlRU9GCg==",
    "club-odyssee-lille.pdf": "JVBERi0xLjQKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUiAvRjMgNCAwIFIKPj4KZW5kb2JqCjIgMCBvYmoKPDwKL0Jhc2VGb250IC9IZWx2ZXRpY2EgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YxIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKMyAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYS1Cb2xkIC9FbmNvZGluZyAvV2luQW5zaUVuY29kaW5nIC9OYW1lIC9GMiAvU3VidHlwZSAvVHlwZTEgL1R5cGUgL0ZvbnQKPj4KZW5kb2JqCjQgMCBvYmoKPDwKL0Jhc2VGb250IC9IZWx2ZXRpY2EtQm9sZE9ibGlxdWUgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YzIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNSAwIG9iago8PAovQ29udGVudHMgMTIgMCBSIC9NZWRpYUJveCBbIDAgMCA1OTUuMjc1NiA4NDEuODg5OCBdIC9QYXJlbnQgMTEgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjYgMCBvYmoKPDwKL0NvbnRlbnRzIDEzIDAgUiAvTWVkaWFCb3ggWyAwIDAgNTk1LjI3NTYgODQxLjg4OTggXSAvUGFyZW50IDExIDAgUiAvUmVzb3VyY2VzIDw8Ci9Gb250IDEgMCBSIC9Qcm9jU2V0IFsgL1BERiAvVGV4dCAvSW1hZ2VCIC9JbWFnZUMgL0ltYWdlSSBdCj4+IC9Sb3RhdGUgMCAvVHJhbnMgPDwKCj4+IAogIC9UeXBlIC9QYWdlCj4+CmVuZG9iago3IDAgb2JqCjw8Ci9Db250ZW50cyAxNCAwIFIgL01lZGlhQm94IFsgMCAwIDU5NS4yNzU2IDg0MS44ODk4IF0gL1BhcmVudCAxMSAwIFIgL1Jlc291cmNlcyA8PAovRm9udCAxIDAgUiAvUHJvY1NldCBbIC9QREYgL1RleHQgL0ltYWdlQiAvSW1hZ2VDIC9JbWFnZUkgXQo+PiAvUm90YXRlIDAgL1RyYW5zIDw8Cgo+PiAKICAvVHlwZSAvUGFnZQo+PgplbmRvYmoKOCAwIG9iago8PAovQ29udGVudHMgMTUgMCBSIC9NZWRpYUJveCBbIDAgMCA1OTUuMjc1NiA4NDEuODg5OCBdIC9QYXJlbnQgMTEgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjkgMCBvYmoKPDwKL1BhZ2VNb2RlIC9Vc2VOb25lIC9QYWdlcyAxMSAwIFIgL1R5cGUgL0NhdGFsb2cKPj4KZW5kb2JqCjEwIDAgb2JqCjw8Ci9BdXRob3IgKENsdWIgT2R5c3NcMzUxZSBcKGZyYW5jaGlzZSBmaWN0aXZlXCkpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA5MDIxNzMyMDgrMDEnMDAnKSAvQ3JlYXRvciAoXCh1bnNwZWNpZmllZFwpKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA5MDIxNzMyMDgrMDEnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAoXCh1bnNwZWNpZmllZFwpKSAvVGl0bGUgKENsdWIgT2R5c3NcMzUxZSBMaWxsZSAtIEd1aWRlIHByYXRpcXVlIDIwMjYtMjAyNykgL1RyYXBwZWQgL0ZhbHNlCj4+CmVuZG9iagoxMSAwIG9iago8PAovQ291bnQgNCAvS2lkcyBbIDUgMCBSIDYgMCBSIDcgMCBSIDggMCBSIF0gL1R5cGUgL1BhZ2VzCj4+CmVuZG9iagoxMiAwIG9iago8PAovRmlsdGVyIFsgL0FTQ0lJODVEZWNvZGUgL0ZsYXRlRGVjb2RlIF0gL0xlbmd0aCAyMjQyCj4+CnN0cmVhbQpHYXRtPGdNWWI4JjpPOlNiYVkraDxfO1JnamZKaVo/PTRtLS5eVWpYPWQoQE1oJ0smZVAiUF1JNFl0OU5DckkkS19MV3FPMDY6YWZiYUcvN01YVjsmbWtjRWdZYmlMLGA9c10saEc/PihiXDJFI0dPKTpfPSlrUWZPXD9Pa0EvM1I2Xjc8LUJgPHRlNCo2R01AP0skWEtdK2lsWFwxalUlaVg6cydZRF0xP0FAXDU0WWVKRUhxUDEvN2wjWlJiclBOXnVDVXVJVUM6Xkk6Wk1gci1YZTA7KiI1T3E9Ii9cPFBFc2dOQDI4QkErX1M7R2FWTlo5KihYanNjZ2thR1JGVCQ4dDEvMytuLmVCTGY1YjlORW43dFRTNVRaJUtrUTZoRiMuRV1hJW8qakIyNV4xYVxRZkAiL2AmRGQtOV0vLildI0wpZnVWSW5hTXEvcElPaiY4KGdSOGxXaHBKXTRjYTFlU04hdHMiPW9NJEJGLVEhbWtwWS0rPVNmK0s2QSFvSmk6OidpL0RoIWZTRXM6WHUsIm5IJ0Uob2JYKDhsIzUjTCpTPk5bO0Y2dWwjRCFUNWRHT3EzPSppaEc4XSlKJiliJVgpNjVBTE1WOS1jJDs4L0I9N04kWVlWMVZQV2AoO1xaUzglcE5GcW41VFo0WCtXIiEwSC1FMGxGbmtnJmFXU2Q0b0FIP3Qycl04cCpGK2VWL1RUQiRLMmwoL01PJEtxNkEtUSZrYEZDVzlGdUFrTHFYPl0oOlE0JnAyRSZAPShrZVotbmthI1g/Mj0mdVZWNkFSQVpQKT5DPVNhN2xvTkRFOUJZcjpVIl49NmJ1PjtZJ01CWWY0SCJmSXBFTDpmNDFMTTdhXzEsQnFGWVEiWlYhRWZlPmdLNmVscFFha2BjQjRrbD5VTCsjO2NYXCNWNT91Ty5eInFUTnQyRmlfWUJnUDxhL2lrOkQwbUtEMzhJLCEmbmwkZ21BRl9qOEUkSyJvayVIPCJiMCFRcFY1LGRybXBHVCErOiw+STUsOFpeLmFdT3BXdVclNTRzaWc1UHFhVidiaTg0ViRxI0ZLKDE1aztkX05adVJNRCpKbFYkNCpVLlUiLXI5Y0VnPE4wMlJYY1UiZzBWaEUjZiE1OmRWbFc+N15ocU9IWiUsUydYWiI+TD5DLV1bdXMoNEtsbz02RkB0MixMamgpaUw8byhkXlsqPC09XWVjUCFlcGFiXUYzU0lJQ0Y7U0NtbVhrPilTbHBsPEtGJElrVEEyRCg9OEJvLWQyM0gjXU1ZcmlmPTZzRDIpZktEND06SFhpNWNQMmRVQjc0RGlKaENAcSUoUTosaWhmczcoPWY/RE4iPUI/YCNhNDokLSlFPTtEZVo8QEYndFZhIVpvMyNrYVtQQmU6YEQkYVJxVD5zY1dyIV5gSW1tOWM8JF9MQ0VObTRSVEhaaGFPI11rUkMtXSQ6LmpNPzo+dDIwLUtZTlcsKSROUikiW0BNXjVZLGVJYilVZjZDImcyNEc7bTxXNiFgKzEycjdDai9cWUY4YmVGO0p1cixWME9RRkpfcCs8Z0pKcjpiUFttcDMsJkcsaydFIlFWMTM4P28lMURPSXJAVzRhS1JfJnFlZTMkSXRtLCJZJCNobl8nI0ViQVcsX09DdXJVK1ZybEtvTWZuRUwlZCE1PEJhKkNlI04vMl4hVFAqQSZPY29UbHNHMnMhTzhmOyVTJ0BqYDljWDcrPjwkZFdxQyREcmE0bCElbk5fW1hBWWtJZ1g/PDosRi5CXmBnM0ZrQkZLJlMwIzBraV0jZClCU0pkI2ViWW1XJjYhZlJlWiFlTjg8cjVMWC9kaVVIMHMnIVglJlBtYUdmaUxZUktyPj4rdDpjJV0pJG9bMHErMy4qR1dvIiclXXFwQjQ7aG1HOkhiLmxZNkJDM3I4TV8uUkk3YyV0PGchTHVcWC5MdW8xYD5VKSpHXGwsQj8nOmxWIlVQT2hSOFAiOEVjXnVEPFRoVVFNdD8pZlNnQkdlRVptZyleQF85UGdrMklPX1p1TEMlQXNfJjs8aj80U0loWmgjJSgsSm1OPkUkMEwsOSVQKClUQlc6UT5YaG82QHI+SidGR1pfKGRqRyloPl5mOCpIZCRmWVQtbzxwWTUlS2dLSWVqJ1prbCwmZFNNUVE6LmFNblssZUpgPUsnPzpGWEhXOzMlaVhKbi5vdCIlP150TVwoSGMwJmE3N0ExMFVUNWxrUmBcLTlbXTFnUmpXQEoscVFePjxEKVlocT4oYz5IWlhCL18hK0peNi44WC08NCxBUEZQZ0NRNEc4TmpYLWtkNW9pP0pjMy1hI25ZckhVJXJFS2pDZlpkLkVCTjtSL1c2b0ZOWFFLKzY+VFQpSClpVyM3XGpIPFs8QUYhVUdnYHV0dDJuZiJYIzMtKHRddFpgLmhGKlBVU3V1W01vRydlUihuOmRqODskWFRxWWBJXlU7OzVVUydzQCFYbUAqY0N0NUonbFZvLSFnS0NoYW9XYVk8IUk2R283SC5RJkc1SnMxQnQxOGU8aHNDI1w9NzcjOzlwIXVsMmA4Wm9SVTBuJ21ZMydbOD8vPT9yWCctYkFmNChQPSRIMEclXTZgOnRUNCNjOylUVW0vSkY8ampLaUBUa05YS15hXCIuQzpsamg2MzlwJF4lXF5lXkNacG0vdFRZVHFXbS9pJSg6ISUla1BYblspUjJhPFxbdXFPPUo3VTxwUD9FOUtDZkM8NV9DQDtzPDBaIkoqZnArRDtgSGtPQ21LbmxdX2JeUylHc2ZsVFBHTm1gUktWamJRQUZqVG5pNTZHJkBNRWVgXUpOXitsOD0maUU8dDJnWUBDVG4vNzElJzw9I1BKayhVIkFDbixCYjM9SiJjLiFkQTpARSZOQVhqLmxkOjldOG9fcD9mLTxXXEcnLEhOVk5uSSIzZVk5a1M2ZE5bcU9IclFLJmk3RVRsIUorXCxVQj1kcE0pLXVhO34+ZW5kc3RyZWFtCmVuZG9iagoxMyAwIG9iago8PAovRmlsdGVyIFsgL0FTQ0lJODVEZWNvZGUgL0ZsYXRlRGVjb2RlIF0gL0xlbmd0aCAyMjY0Cj4+CnN0cmVhbQpHYXVgVGhmJTctJjpYKFRFQSddLV89OSkwRW9RTSRELSUtNVtXU15hQDcrQDdNS1tXS1BUa3VrZF9DNlQ3blQnXGEqKlJHNk5BSmdKKSdTK081SmJKSkolbCxyJGAjKGY1cGBUPmxtUylvLUdaR2k8ODswbkRqR2NtPkU3OFM+TzNnLz1rYEFaWVojUVZEREo0X11wNSo9ImVOSlV0Yk5oNE5xMj5LS1ZpdDJzajRhU3BuMGZCMV9OJlVKJ3AtRTJhaSgyaVoua1lgNDg9ckZiQSlXTU5ZKEpRdEslSm9CanNoLTE8NWYzaEBsVTltcjkyIj8hciRMSjM+R1I8dWYwYE1uYC5waDRTPVJgJ19gKjNDQ2U7YGJKPE0xbmFkYXNOJ1hHLCc0REoyLlc9SzEjOiknNT5cK0tQW3E1XDlqJ0xIRjZNMl4tNXA7LXJTVz9zMic3YTheLD1eQCx1ZzNJaEdXI2RaRG5mNldHVVRzJ1hsKF1hZFk4ZGJlSFlFYTBEY2wrdGIlN2FRYSx1XjMySzlnNj1gamlZc1tpWC5Day8rKi49YksvOkNGQW1vWGY+UGQiVm1fQEtnbEheX2lcS2NXRVQwRFtiRjowJ3RKKGQ2WDRdZkFAJmMiZzA/WW46Om1PNUg9OWM4aiwmbEcuckJtIXJKM1BONlFtdWRdUG5uNmROYj5tNicvUDQrYFciUFQzXk5LLTNxWV5TM0tVJXBIbldEZ1loPWo+TzhJaGxNckNVLjNsPkhyJ2RUUW4yTGtTO0Rpcz8oZUEpMjJtYGpEbW5wUXE3M2B1WUtsY2A+bnExR0dMal4rSDgsaWlGRUluUy1BJGUia1VJdSFWSjVEPCYwME4wPFxWX2BXaiQ0UCoidF8kYjY4RS9jL2Y6ZG1FS10rJz5tNjwiPktYN2hnNjBeUCRrSGNeNyYsRjVfcmVOVU1IPiZkL0FvKVJDaWZkIy44VWktTGVAOFVrIkRtQz5BOkhVKVNwKFREWidsLzMyUzRVMFgpJTdVUU8iS1tLcE9fLXNHKkw2NXRWX0BDWWZtSEE7TTFNQSc4NFZiZFlFWG0mU2puNDg4SVBrXFxaVS9pZnE6Xl9PXihhPXJIS1VbT3FPXnJVQyohb1k7b0Mna2YrRFVgNlAqbSFdOWxYaUg3aXRDPFUyWEJyNEY9a0o2LzgxYWJybEQ+WDA8SS0tXEhJa1AncG8wYz1UQEVTMW9dMj8nS1IyM3VPI1NQTD0uUEBdXm1zXz9uPzxMKCh1ZUAiRmJkN1toa1w7cE8rXSJBPms1OihCPkQvVUJwaU1QOlhjcVFSKFBWVlI4UUozbTRxP3U7Mk5fNTNeKEk7YWVIZzkrUGk2O2lta1NmOiszR20mc09QTSVgYCo1XG87O0E3Y18iM15pVW1tQ2FfPzk7LW1VdCd1VU4hVXIybSdKOHQmaURgcGBPbzxWIiYhWTZWOWhyKzwzKy9UKStLKHVuIXNpXGUlNkdHPlUudXAiP0FXNUdgYXRkPWtkdGhgLDEyUGsmby9DbCpScUNBOidbMzs/ImpdQVM+bGtQdEREKzoxQWMnKjkxc1c7cjQsZSlYb1ZhJ2RhWEBncktLKS5wNWg1RipzOVhjTzVAJltjKkppTkIoKTpXNT9dJTl0T2ckV1tWTTU2YDZdJE1Pbk83OF9IJS0xb3NAMTs4NFdXRGBCa1ZCZCRXNzJdITQoYCokdW5qJGZDJUFWO21QJGhVMSNQKVpzbFxDOGk3QktKNF1QRHBmczU4YkpCQ19aT2kuSGtXT0pRLWBMTHAsYjcpUXU1KEUsLyx0NiVaNm4uc2YpOU42NElvP2JKa09QNFRYcm9SakhtUDNJUDZfPEpeTEsjUzZAQz4pLj1fVnRyLHIia2MnQ083X2lJVCYla3JFQSlUQCc8Pl8+YXBvM0NhWCtgQzxzczU1YFZjKmNgJV8wVGUqUGlhRjJvL0pxQSI/LkVYVkQxKzlOc1UlJl05NHQ8PjJZYGspbDNfQjJgcUw1PU02Rm5pR2ErNixpNTc5O0piP0Y5VmkmYVZzRkBFXFJpZ1VVZXExYCdpKToxYj4sOjkoPG8nYmlsaSY5WFg4QGdhUmU2ImdMW1dIW1BEc1cmQkJCQWwsWFd1MC5saFxMQzdnMzFtME9nJSxOa0oqYylxaVsoaUFBOlNbNklMKnBRRm5tcCJGYW0qdFtUOGMyUkZVNjYlM1BaaTo2NyxLZmdGTGpddVYnNnFjXi1KQmc9Mz84c2E+cT4/YWdMRygzZ0hIWjE6OzJUdDM7cTJ1NHAmJyRwOGdDRzxZcS42WUxSTz9uMlBLKCshQztbb1M4anVAVGJYWWhCYUhAcU9uVj5OcGlrL0YtJGBCcCNVIWEsbi88MFMhS2plckhgXiwuR3VIJzJTaWdMKTZnZEdHTCZWQ00hVlthIkZqQC1pK2AnIkBaRy8kYWswJWpjZT9AZTB0S0lqNExlVClJM0VwcC1tW09KTXNYYko7dEpqL2RnUkNPIkUwUEtaIlVZQ2o9V05WPi1yMmtOKlI1R0NwYzE3ZyNmLGdOMS4wWEYqdWV1JjdaMyJNUGE4NU07O1l0Yk4zMzA8RExkWUdHXiNMWCRVSFJEJSoobnEvdWMoPFhicjROJ25YUFw8OXJTMEBhTms/LzA+SFNHbjFJQEY9Qz9ZQ1NiRVNiVStvJ2xATzNUJ20+WF1Cc1lEW1BHQyJsZUpPY3FkbXFVTTYmaV5VV0gqL29zbnFYaFhDYElGWi1fJDdfMUhlbG9zR1I8IVs3NF88SlQsMF88Q3FaXGtqY1JBMzg1W01xZ2pobypSOEVrK1QvJE1dP1dTXisjVyUzb2YhRmdCUjVmQzV1QkQ8TyNkV29zaEVvbm5XNCMqImpCYShMIjguVm1haFxmLDhXRDxzWUJkMXE9PShSPFo/UHVoMkFlRj44WDQ4XF9bI3MxV0NcciZoLlwiaCFJNyw3USNJT0NrVWxWU2xea2UwUixiNkteIiJYRj5Aa2x+PmVuZHN0cmVhbQplbmRvYmoKMTQgMCBvYmoKPDwKL0ZpbHRlciBbIC9BU0NJSTg1RGVjb2RlIC9GbGF0ZURlY29kZSBdIC9MZW5ndGggMjIxMQo+PgpzdHJlYW0KR2F0VTQ5bEsoNSZcWk1vVzExSk9tImdsUD5NNm8nKFJZXjY5UC5vUmM2OXUuPiVDOUQmJzx1LUxOWSZbNS1MdVUnXW1fRTYzYDBdV1B1JD9JZUR1K1swaUFfVmlcWW45RCRbWHJVbz8qTU5UJGk8OD88ZGRHbTw3cXRUPUoyYXIxOTxTam4wamdxLF5lYWcuTllMXz9dTzY/KT1nViYvbEBQa1srPy4qNVA5WzxHayllRzViWFlsWkldKkVaMWQ6bEF1KE8ncThZMS1kI1ErbTNXKmhWXWUwQzhMOXIxKj1BP18kZTxcLDw5JyYnalU1QzBQck5bSV5pbTwrR2pIYE5vIUJvRkBwOksvSXBNKy5SNENEIjFvZ2xSbCM5PkxoOVg7PSg1J2xqT0o1ZjJtNDouO1c6NF0maSwuOScmYTZPamZrVE5KOkxfZkdOUEZfNjVlSVApISZkQmczaUdKO2ZQOzBALW0wWEA6MUFDXTMuW11vPDs5WTpLQWldaUowNjpMP05oZXBIYUFcdEtRZGFRaD5VN1NbcVwlM0xBIj4yMVJpZTRycVBTWW5Oc2BcYV5VOlVlbkw6Mi4xQW1EOSRQPjI+Z3M7Ymc0XmBCbUpwKz1bLXJpUVFMP19VKkNVVU42V1UzVV8oIz4rXXM2RTtycFlyNENEbEY7RylxaGNnbTZXODs3OlM6KzJgUERcMylEay8jUWg0RlktNEJDKy1LKSRFNkZFXEVdSlNUaSdWRDEuSVhGWjM/O3BLQlhXR2JiXy5aLGxrSmUwM0dnL3Q1TCwnSVI/I1ZiWSwvLiJscFhEMSQtQWVcR1Y5VWIsJWx0MV0uP2EjQUY0b1JiITYxbV1cTyJFX3AlRnMjOEdLVSE4OjNBWCwvU28sQHJOai42NFdmMUEuPSxxW2RCQSYqXzI/UlAmT3RgPVMvRV9nPHVbMDszN2VfZD5mLC5SNEBWJU5FY3JsTmdOZFkpLUEyMHMrWG4rPTdURVNtOTREb25vO2VBdTM5P29AK1lQLVRoM3VXcllvUFJmL1o3WiVDJ0dFdExhKSQkTVYuUnEhW0M2ZUUvTEpDJTMiMCY8VVBaMUJSQC8xRmRIaTlUalllJUJlMm1YSEQ+NEM7Zmg6MzYyIUUvaHBDNytPNkk0UEtUbkFnUzIvMzZvWSdcKlohPHV0Vl48TEAkZzMqSz8tX29SNzdpN2lvc21PNzVgciZuREUtcCphO29OcXBRUiluSFdCI1FkJT9lJFg7VTE3X1VnViYrNEc8P0c7Sj5AZSohQWRQTD5UTF9IOjRQVzdSbEcocTtJOz5IaEtSTlA6U0liT1I3X1hjTUluSGApNFsuTnFnSSdOWF5sNTIyVkAsSExuIlhWZDFYP14sZVdRVDBjZTdEczIlKyw2VzEzI01YKl9fb1RYSixtSUhRXWYsck4uLW4vZSQnUDwjKF1pT2tuSFkhIjNDTClkQlFVKDViQ0w+a2xZJU5pJVY4PkxXSVkhMUd0Z1VvVnI7dSFsRWFLVC1xaXM4Y0xOT0lFIW0oUjtPQi0wcF9iQVkpRG0tYSIoQ2A/QjpJJjg5dHFZIypDc1FrQ2JCJ21ZWm8zKV9CXjxYLExmV0QwMi1YWV0jaSM+SSdjVGdyZUMua2JMTywyM2k9U0ZScTxCQlV0dE9YaF5DbW9kb2hPQDEwc3MiaTleM0JCZUc2OEFUPG5fSFM0WDYjZSs1KylqTmRROVRoQjclNDU5MGtkT15sU1hfVDVybDk6XV4oVE9CMSw/WUhhOVZuLztDRSdvV15FRi5NQWZVME5xKyNMV0xuVkJfNGxPRUw8ZiwjYkZbUEk+TVFRXVMmKHNBWDBcMVRjNUxrXkU5aV8hYGdwU2s/NT5EP0QzK0cwVGtEXiF0ME5TP25iWDw1YWtXIVEqIlUtXnFTTWxQVHAxLVxVWz9Mbj86YEJGaEJlU15oaFFPMS0pPVFnaT02MEw9IzhBKjxoR0pDbXJIc1NEWktpP3VKOSlic2hyIU1dbDVcVCgwJ3ItOSdsTHIlTS5TIkpPMGJgRDpCbzE7aVUraVpVOkpUTmRxWVcqVmRhUWQpaUhTajtQTVQrJ0VIamA2NVVdIlo/PDA+WUY4JSVYLTM+Um8lVyZ0dUUiVy4iNSE8OytbYF1zZisoO2AtOiwnTDhPUmBwZmVFQygwMzpyNFR0W1MoLysnKWtjOTdmXDstc2pCKFFsWFFOWUZuP0hES2xEQjdYYyxmYEQkdWhEZT8tV2dkREpoLCdGU1Rrc29bPy8oVCEnIl1MKWJhUSRUc0Q+ZjBIPV1iTEhkYWJBUzotVkczOzBqdUlkckYzYDMpTXFCJi5VMURoaidPbltvLkkpJEY9YjdqPTRmY1NSazM7amouSzYmUTZGL0FiKV1rWSU8RyoiamQkckxePGBKaCxXR0o+P0dZLi11KigkKik7SFBAMCZDJCVtXjdiI1lkMjRTbGxmRU5sVT5wLW1VSiopOWVbNlFqbjhSPS1uP0lGN0JHRCVSWlw2LlMzQXVWYWMiMXApYDEjbkY2KkZlc1w6QXFGcVl0RideSUZNZGUuUVUlQlcpPU1pPjkiLzlzOUZFU3IubnU5QE9UQFI7PjYkRlQhOloyck9zZGs3SipOPGxOJlZLPUlKWG86Ik1YJCxKOyZGQj51WlYwXklXRHM8bEldJHIwTjM+aSY3V244Y2clU3NUNi8rREpbMSR0LT9Qajw3PUwuYTZpam9scFdpVSJIPkNpV2FJSHN1TUVpLitDZllVOmRpR1NLPDlRV0tkQzJcQDptX2M3JEU0TDUxJjcrQ21HNGpWQjRsYSFxXCciWTVXL3BwZDghWUFeYW4qXD5dbiFeRzJvYDsoNm8/PEEzR1VnVnUsXjhPO1IvQFRBLzpBbWJVPz5uN2coMW1wcGNeZlNsbjhAMzJ1YHQyOnQ0X34+ZW5kc3RyZWFtCmVuZG9iagoxNSAwIG9iago8PAovRmlsdGVyIFsgL0FTQ0lJODVEZWNvZGUgL0ZsYXRlRGVjb2RlIF0gL0xlbmd0aCA4NTEKPj4Kc3RyZWFtCkdhdFVwOWknTS8mO0taT01ZRUZEIVlbWDtPQWVSVTg0W0ZxaVNNKFxTP11FIl4kRyNUNy9xW0pmLS1xPDw/aVlXT2g8YjNTcEJJaDhNVC1OTlcyRUxJKkhZW1JqLSs9UVVqV2cjbCtsZnE4cklZaC0qUlpBMjtFLC41JlZTUCJtQidwPl1xMzlSIWJpTWUlZFppO2debXImLW8nI1VhMmoocjAsNzVQY0YoRz8sOitTa3ApZGc+VE5EIk1UN01eP1gvIUhNLWYsRmEnciI6OU9RZUUxQ0FcUF9FMkNjN2J0X1luWG8najNAMTcxcihmX0pncUVEQ2s9I2kpYko8ZSszKypPQkNxYS5WJC9tJ1s8RW0zYz01MEdpTSNHQlhHIydOJkdxSmJCUSNHKkA6NiZIO0EkaiUkOVNUJG8xU1dtTz9sJiNFc05PK3EjSUs2XyY0RDMvM0ZdUiZaU3FHIV44ZnJJUkQrdDIqOjNtNiU0JCMwTG5XYzsrKypwXkhnI047WGtHM0wpVDxHO2NtclxQcUByMlNeY3A9ZmpLU28pKXRkPzxbX1YuTF9uYj4tR2FoXCY/XD9PN0lCOEQzbioxJkJJOShgJiMvIm5zLHNDLiJdRFRwXVE9Qj1yNlFxMFQkKipQVTVrNE9OR2MvIys9SlVyLzsmZCgtITE0IikuWGsuWnQ1cTclIU9Pb0kpbE9FYjlQbFZkNis7cWRJXFo7SiglK1pDOU9PRzZMcFtIJihVWG5QM2UlNlhecllCQHJdOVlEL0Y2aTtsbVc7cWQrKUNMMkM0Kl9lZm1pKTIlP0xsPGMiJWxScCprMlhVQyZycGk6Ljcuc0NoX3VHQDZuQyFvMUxhQltsNnFsNVwyNWU7QiVUMF9fPGRjZmEia1spYWgwXy5iKnBpKFVGPCI2KGkyckgjIzMxWD9FZD5dSzU/IzYvXzc2Ty9qRj5qWis4NThhPjk+M1lYYStvbjRZTllbS1p0V0JgTUcsVWdpMjtKM11hPUVXaCswLzk/KSc8P1ArNkJcQFcoTHMiO3JmQSdASVwsdDNESmxwUGUzSDAqOmQlX1EobnIxVFltODZNKWpIYz08LVFtNVkraThIMm9NLGBDKF9UPG1SOiFFVn4+ZW5kc3RyZWFtCmVuZG9iagp4cmVmCjAgMTYKMDAwMDAwMDAwMCA2NTUzNSBmIAowMDAwMDAwMDYxIDAwMDAwIG4gCjAwMDAwMDAxMTIgMDAwMDAgbiAKMDAwMDAwMDIxOSAwMDAwMCBuIAowMDAwMDAwMzMxIDAwMDAwIG4gCjAwMDAwMDA0NTAgMDAwMDAgbiAKMDAwMDAwMDY1NSAwMDAwMCBuIAowMDAwMDAwODYwIDAwMDAwIG4gCjAwMDAwMDEwNjUgMDAwMDAgbiAKMDAwMDAwMTI3MCAwMDAwMCBuIAowMDAwMDAxMzM5IDAwMDAwIG4gCjAwMDAwMDE2NzkgMDAwMDAgbiAKMDAwMDAwMTc1NyAwMDAwMCBuIAowMDAwMDA0MDkxIDAwMDAwIG4gCjAwMDAwMDY0NDcgMDAwMDAgbiAKMDAwMDAwODc1MCAwMDAwMCBuIAp0cmFpbGVyCjw8Ci9JRCAKWzxjYjFhYzdmZGZjYWNlYWRiOGNjMDBiMmJjNzc4ODYwOD48Y2IxYWM3ZmRmY2FjZWFkYjhjYzAwYjJiYzc3ODg2MDg+XQolIFJlcG9ydExhYiBnZW5lcmF0ZWQgUERGIGRvY3VtZW50IC0tIGRpZ2VzdCAob3BlbnNvdXJjZSkKCi9JbmZvIDEwIDAgUgovUm9vdCA5IDAgUgovU2l6ZSAxNgo+PgpzdGFydHhyZWYKOTY5MgolJUVPRgo=",
    "club-odyssee-lyon.pdf": "JVBERi0xLjQKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUiAvRjMgNCAwIFIKPj4KZW5kb2JqCjIgMCBvYmoKPDwKL0Jhc2VGb250IC9IZWx2ZXRpY2EgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YxIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKMyAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYS1Cb2xkIC9FbmNvZGluZyAvV2luQW5zaUVuY29kaW5nIC9OYW1lIC9GMiAvU3VidHlwZSAvVHlwZTEgL1R5cGUgL0ZvbnQKPj4KZW5kb2JqCjQgMCBvYmoKPDwKL0Jhc2VGb250IC9IZWx2ZXRpY2EtQm9sZE9ibGlxdWUgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YzIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNSAwIG9iago8PAovQ29udGVudHMgMTIgMCBSIC9NZWRpYUJveCBbIDAgMCA1OTUuMjc1NiA4NDEuODg5OCBdIC9QYXJlbnQgMTEgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjYgMCBvYmoKPDwKL0NvbnRlbnRzIDEzIDAgUiAvTWVkaWFCb3ggWyAwIDAgNTk1LjI3NTYgODQxLjg4OTggXSAvUGFyZW50IDExIDAgUiAvUmVzb3VyY2VzIDw8Ci9Gb250IDEgMCBSIC9Qcm9jU2V0IFsgL1BERiAvVGV4dCAvSW1hZ2VCIC9JbWFnZUMgL0ltYWdlSSBdCj4+IC9Sb3RhdGUgMCAvVHJhbnMgPDwKCj4+IAogIC9UeXBlIC9QYWdlCj4+CmVuZG9iago3IDAgb2JqCjw8Ci9Db250ZW50cyAxNCAwIFIgL01lZGlhQm94IFsgMCAwIDU5NS4yNzU2IDg0MS44ODk4IF0gL1BhcmVudCAxMSAwIFIgL1Jlc291cmNlcyA8PAovRm9udCAxIDAgUiAvUHJvY1NldCBbIC9QREYgL1RleHQgL0ltYWdlQiAvSW1hZ2VDIC9JbWFnZUkgXQo+PiAvUm90YXRlIDAgL1RyYW5zIDw8Cgo+PiAKICAvVHlwZSAvUGFnZQo+PgplbmRvYmoKOCAwIG9iago8PAovQ29udGVudHMgMTUgMCBSIC9NZWRpYUJveCBbIDAgMCA1OTUuMjc1NiA4NDEuODg5OCBdIC9QYXJlbnQgMTEgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjkgMCBvYmoKPDwKL1BhZ2VNb2RlIC9Vc2VOb25lIC9QYWdlcyAxMSAwIFIgL1R5cGUgL0NhdGFsb2cKPj4KZW5kb2JqCjEwIDAgb2JqCjw8Ci9BdXRob3IgKENsdWIgT2R5c3NcMzUxZSBcKGZyYW5jaGlzZSBmaWN0aXZlXCkpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA5MDIxNzMyMDgrMDEnMDAnKSAvQ3JlYXRvciAoXCh1bnNwZWNpZmllZFwpKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA5MDIxNzMyMDgrMDEnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAoXCh1bnNwZWNpZmllZFwpKSAvVGl0bGUgKENsdWIgT2R5c3NcMzUxZSBMeW9uIC0gR3VpZGUgcHJhdGlxdWUgMjAyNi0yMDI3KSAvVHJhcHBlZCAvRmFsc2UKPj4KZW5kb2JqCjExIDAgb2JqCjw8Ci9Db3VudCA0IC9LaWRzIFsgNSAwIFIgNiAwIFIgNyAwIFIgOCAwIFIgXSAvVHlwZSAvUGFnZXMKPj4KZW5kb2JqCjEyIDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoIDIzNzEKPj4Kc3RyZWFtCkdhdFU0Z04pPUQlWGp0UV5lNztWUGNRMTFNb0U9VGZ1XkBkYiYhPytWZzMhPy9cQV4vOWZIOnVqUWxPP2g3VWc9TyZfQ0xiRWc2c0R1Xmp0SjUuUTA9IjZGbVJjcT9jclZNQTdkWW5aKzhpTmRbKkgoLS9pUVstOkJmcnJMPGtDSzs7bkNaPVhPO0tfaiNQbWlqTDJhTyF0NEAsJ1A7KHNiVT8tOkAkXCguSSdbIXAydT5jKyslai1AXkBITD8+UFpbV3BabFtybDFTV0Fjb1loOWxpLmdocyRfXTUzZGFaUms+KS0sSis8MU9YRT9eWktgSyFhO2U1NEUySUswbGNRQTJcby49OEhrWTglVVpgNXIrXUVba0FHRSFcJ0ZBODVWaDZqdUYtWTVzSV9qbFtvZj11JylrcD5gZixadE5JIVVzXzVBZVckbyYrJmxiZV1fb2RiYmYuVl5ESENzKTtVZykiNU0kdEpdLDlfQXQqXkdGZ2YmUVpCWmFrPnNhT08+PjpfQys9VigoOF9daFlLNTpUO1RVIjcyQVQwXTAyPVgmSV0lOW1fWj5PdWhXRyhVJGdWVG4oTTsrTVEuUSU/TGpdXkNYKjdoVmpwV0RiOUk9J0JHRSxMaGxsTSZzYSxOZ1pTTDdJclAyPExebjs+N1I1UzowbignKkRyOFNbUFw8KVhYSVtmJWMjXDNiQE9sP1FrYVxxaVhzYjkvZyFHTEhrJWc9MnEtKVApPXM6NF1QWmBjK2dtNTQlQSwsVTVva0g8XS5qS0U/TCwrYSJzaU9vcSd0T0VvWColRTklP0JHKWBacFI+dSxfb1MwVk4iLF1MWkltWnVZaGVhU10nWGkuRW4+J2JsRjNkcG4wYWlsZkp0TjZDIm01cWYiVFw6STxEKE9nVU4iPCVNSmNvTE4/NlBTKVY7QldGNWtodUIzXGpgYCk4LEczNjAqP1k4XyY0bEBWSEkmLlZTKSljTiwhK206ImZpX0VMTU0kND9yIVdjLDhbaUUpVComVzZycnRoWHJgJUtGI1Neay48MjNqKVxlOjh1K2ZdWmM4KHVVYjtvSUhHQU1oMlRGaF4zQ2BeaTZaU3FUKDsnUVcxQ2Y1QzElMiFmR1FRcjMwKjc5U19lJzpucXNCRmc3R01ZNi5kPztdNzg+MU89MTBcInNuSkc1ZiRpUDYuI1NiPEcvQitFN2VSLixDLTkjSUQtSGRFZSk+QC0lMENsODVkYkdCTTtLRGw6MDtBTyMnOmhnZj09R102O1I3SChYP2xVbG5YW11URj8nbmQ+OjlaXlckYDV1LSIrbSMwKk5GYz5BcFI6TT1PZmBZIWY2PS88YjBgUiprXyE2ME1OLyRuM0hdKHJWJS8zMkZMT2NoSy1ZI2k1VGtEZWQzKSxpXm1RKSFgZCMjVkU1L0dRJzBEJGlSSG4xZDxRMyZMPTlvTUpRVTg9Rl4nX1FyUFYhNGMrI2FjcmUhLFQ+N0FJKnFNOGw4JWdzZkY2I3VfNmA3SWBMPDBlZzY0YlpIN2ZoK1YlPi8uNi5ONz00ZE0zRi85W2JJNm5sNkI3XzNyOUZENkRZQEV1QVVdTyRLYD07JUR0Tzs1MlguOkFmJyxpV0IqSFJ1bnNyPjBsVS1HMSphOF8nVycmNT1Fc0slVlpsNVEmVzg1P1AkMGZfakhzRXNGMyoiIzJxWmpaVS1HPkBIWmVuJlApKzljUTZxJmInYm8vNXVPYmArXVlfUEp0N2o2OFcjUzhoKDNEUE1uOG9DUWNnJjhnYWBJLWVSPWQ0KjNvX0MmWltUKWdXIWhCP0gkPy9tXnMldFlkU200VTdtIjJPVS1xSC1ETSE+OT9rVWkkdUlEPEwhcFpKZyxpWVtrK0EhKF1uPUgmTDFVM1YmcGBcZE9EJkQjVWZSQ3E0ZShyTmdLWUU4PERQVil0RSJtJjEsRElRVUJta1w/O29yIUUlKjVfUE5saTE3Q0FeT1p1OD9CK0BXaz10RGw4Nzw/VGhWTjVeKz0lVEBTMVBAVmU+QkNJIjQzTF1aViNBNT5CPyNDW1ZVa1gwPGs6aUREW0xtTyJOMT8iQUUyYiE5K11IZTxPRSIzQGImOU5FMig6LT1vRiMpc1VuOWouajBiUDZxS0tmb1UkVTAqK1NDY0lbJTBRNGpoMmU7ck9NUV5CanBjPVhkOWtjWD1wOklpKGA0YSxwVTBRQzZGYDAybHBXMWdXYyghSW8+TGVVW3MrNXI+ZzVKYnA2WTJsbVohTWchQm9iZC0+OWVSODhFKU5SZ20lWVROSHBQQkAyQld0MGB1Yz9xTUFoOiJlT1cqUT0/X0w0RD1UNCIhOEZqMCZeZyFMTi5yRCs4O2xsMiwpNSZOX2AjYUlWLV9vVFBdN0YvNl9rZTwwXEw0SjA7PysxK2trKFN0L2M4VzxabktAWzNoYkoxSE5sIyx0dSEzXSpxcyZtYUouMF1rX2FwNEZbbloidDQzRlErZFlOOCQ/LyJhVC5JIkFvU2NVYiJDV1MqRFFRKUIwNyYtVUU+TmdQZTU3IlZlW11tc0s8MisoU2xFJ2tjPCdoITVJIWMyJk01XGlhXVVGY0k4VSk+Q0k7TTtDOipPalVnL2IocChEbzQ4RENwMzdBZFdORnBCOSR1NmZLY3JDLUp0RDQ9KnNNPiQ9aGRfXDQ8UEpBYWlMLU9FO11wWTIuPz90YGg2YUc9Lig3N1QiJEIkQCNBQjtUYyMqLUNyUWIrI1EtXChvYlx0UGchLW8kNytbYFU4ZDoyPUJZKmZyPDc/KVwwMDMyNC01bV1mbnNLck87dCduKGRgLkNQOj06ZFxkdEc+QzZ0bURFbm0hMzRpPXRUJE1gV2xJaihOMF8lcXVYNW0qbmBObmlWLyFkTCNBWE4jLzZYMjlXOEcmNSxcSTpBPXUlWkJtLlZeXHUiaWlqVlo8KitXZTJKZU1AXDpeQkUmX21eTWBDP1A3dEVdcE9EODVWRnVsKCUwLVskNjNJby9kZ2ovJTQ5QFs6cko3ai02cWxJSWEmKldmO3UhUVhAc09VXGZNP2psZUljMV4mKjJyI15IMEAyYDNMZGdfQFZAVEEhX0ElNXBfTHU/L1QxLWEqVWFFLy0qR1RZdW1HNExyMF07TEBYMnVhLVtGVyZtfj5lbmRzdHJlYW0KZW5kb2JqCjEzIDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoIDIyMTUKPj4Kc3RyZWFtCkdhdG08Z04pJSwmOk1sK2xxPkFvN3JgZWRBTFclQi90RmhBPXVUOz5kdUljWCY8W1RXK0FFRUloNjY7OEp0c1lyUCpOUTspLVtsLGg4UydSMm9IOWQ6WCRSYiFVc2gyNVB1S2dTQTk0Vm5hTD8jZGdMQ15JPFAzNURLaCwoWmRDJWRpLmptSSgvSUM1NEdWKWwwQHUrcC1tQSM0U3U2Mz05VCgjTFU8YipNQT1lN185IV5NT1pJLExrQjVqT2FjbzZzMVtCUFhDKT8iNDRmM0N0TkNdaDUiLF9HTEtEblUtLVM/S2MhTGZuR28uVlJqUlhVPUcnPidMUCswLzM+RzorSk5mRSo7XChyNTtlQztVZ11bZT5raTtPSlQtJFdFQlAmRjU+MmhZY3EkNSw7bStwYiE5JjZXZSY9Zio8dThMRUduTypaXztjdGhubSUtJF1rPV1pKF90YHBpNj9MKUElKG8nc0AwOW02Yio1Jkp0TlAuUzokTywtUjhRQz0iRFYmWTI/blNBY18/cmUkWyhscnByKU1BcVVoSD9KMUlaN05qPWRCalNiW1Q2RDFTVXVhR1lzUDQ/aDxwKipyYDVUaE0iUHA0R1krYk5tMUBraEtyZWp0L2BGakczOjNgb2xCdE46KkRfP1E/PjEnaGgiOClZXUcvNTdeXkpiaC8zVnVBQWN1aUIqPjgpV3I3T05xOz5faGZqN0J0Rl1iR2JwXkdWVWMyMmVSb0MpJyJRLnFKaWIkXCkjZy05UTA1UFo1bjcwRXREbGttXFg3aUFbXT5ldE5hLiIsMWdjKnIzWGQ6ckdQPllHNlQjUSs0WWFrX0whRG9sVEdrKiJfQmVrdD1mTz0wQTpCbmk1IkVhJEZKUTktP3RJWTssJ1tRLGA/UmYyVmg4RUQ5VTBHXShWTUhlaCdBKVVUdGxsKkBpUE07ZjdWSEMvLFcrV0s1O0VjVFk4Qy9IQzpHL1NBK1Q8QGtBSDMiSFc4OSVmPVJnMSNKRSJvbyFcakc7IVojTDUxRGNyRmpRX2tQUEM6Y2A0JDJEITokNkNiNz5qKUNgPUcuOVZOMDQzdFEnMSZXZTlccVNRMVBpUF9CZ1UtWWViKCxsWGNXcmhnbEgoYyk4bj5XYklFXC46WD9GbmE9RGgwRUtUKVkoS1QqR0tAIk0hWiwyK0pPRChOYVZZQC5RZUhJXDtKVS1oY1UqIidcTDtITFVOQlhaN2FCKEkrcyYqYF9zKWdTN0k0O29mSiQoLjopRlo2XDxEWXRUWmBASDZjPjRiOyJQKVZrVyJOcy1dNW9kUyR0LUBbYixUQFBeXUdxJWBgb1NWMD9fVzdqTDFhI1sxbF8mUzBXZmxVckooJTxxJz4vUEVgJ0F1WVM8LzA+P0w6TG5iP04uUXNMS0VoI05IUktuIVlfRD8tT08hKEUhR0YhX0AtKEEkRnFyOlM/YU9zXj8zVTcoNHU5XSNcbEU9SyRuSypVR0twQFVELDttOi0scGIzS1pmXVpfKUFIQmVvayI8OkcxIzM/OGxaUkQrR09eOURLMVFvY2RJbGVjb3Q8UENmZyRRc2tpPCQ3bE5xcS8iQ1hMazJ0TksmYDgkKTUrTCZQQVtSYW1ySUlhNVkwNGxHR1JjP2M0UlRIOS5UJFs4XHMrXTtiN0BgOjgyOVBRPih0P2dHZG5lWEBoTS1eSzhTP2lVVSpqTCNiQVBYKipWUFkoP2FeMD9SUEc+PWRMYEpsXCs6WjRhVktHN0xlNmctND5yOEIyPWwrJUQ8KEZBMk9YKTxicT8uRlBcTXBfKGw9Uiw6KWwxTW9IYyFQLTdwZFYnImlHUXVvWDYoOjBDU2cuXWFaOmxLZl1xJE0jRCswRW5pKEJcW080TWtHWUcwVWUrQ25QS1hbMHNvPDtvbE1gLDY7Jz4oMzxPOGJDUXREWlpFQV8xUyElRCwkZDNoYFwjYFddcFJKRkVJTUhHMmlaPlooM2tQYTUyXC5yaSMwJ1pTaUdHIz4iZmEiJ0VWISohSV0lRDgrTSFXSE8oXCMrSUd0JmlRYlVbRyJFb0dKRiYyaC5xRmhLa10kMGE0MGhmY0xtTixMWyRmMS5ObkJPWHRGWGI6OERGa0dmX29GZXBITlEnQnImSyNMJVhjJldQb2lVPWQ7biplTz5tSTIvQywsPDlsWF0pRiRTLSk4THJxPm0sY2AvSywjPCpwNVlUNWBrKlcyV3I/JV89QjNybkE1RllwIjh1YDpYKD5kMU1COSFIXk02Y2IncFxhaytsZjNnP28kXEFaR0oqTCQ/RVpMXVpsTSdsKGtlU2Q6aSUhJj1qVFFCbkomOGxlKV4hUFZSSz1MP21PKjVicTcvYmZwaWtAJDYrRS11ZUA/LWBoOF9IJ2JVaSFhZF9NXShKKmZsNkdAXjxkSDJWPm1RU1FWRE1MP0pRKCI8MDkkKDloN1cxOks4JkNjcGF0O04ncmghcStEQTRjREBwJ1BsLCpAL1glLUwpUVFfJ08/XjdYY0NNUSM0KW5HVjxqYVRDOV46VF4iSDE9KWFjZFVgTzdsY0VSalgqQElwSixadWFqSzxyUGlHIzY7RVYmVFAjIkcucGpLNDhdVFYuOnEzMjFZOVguaSU6Py5oRElKcT00dDg6aTVDYENRLVIpLl9RSUMmLz9kTG4xZWNLVTtIIkJpZHRZTkxwXE07ZjZJRERzLVcsKSQtMj9eL1NBJmp0N206LUJUKlYlYEdRYWxTVyErXVhEcVsuRkVeJltNXm9obCUnXUcxbjtXTnFJZyxgQyZNR1xnb2gpUSZQZ1wuLVZCOlEmQ1klWDtNPXQtJClYWjBgakFQRWVOR150T0BXKHRoXjlqJ1IqbChUczIjL2RAQENbMVVmUCxJQy9JMFUhNSosXlgjS2RXQ0dPIzklLDRhYUUrYV8nUUJaT0ptPXApaDl1Q2psfj5lbmRzdHJlYW0KZW5kb2JqCjE0IDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoIDIzMzYKPj4Kc3RyZWFtCkdhdFU0Z01aJUAmcS1DVVcwYDMkJGVuPj5BbyUiQmZPMiEkQCQzTCMmbXRhVFspZl0yJWJdLCE3SWZJKXBKYSJBZDJtWTFoOCQ2SDs7R05lRiYjYDlVXU5jV3Iyai5zWTtjKj9KITxwWEJRSHUyaWJDY0tPN15dWmg2NjppSWNZPGZbOUNvLS0hZF4/PTY0YjlgXWVjVGFOJmMoJU9bTSRVJT48MkQ7J0FKXzY8WjI1NFo/SDtkSiJCOGslZSwiZlhubDFIKlY+TCl0SS0zbWNdMTslVCZcUSonXS9IZUxVMWswTkQ7Jiw6RnBUNDM5IVQ+Z0BxTE1qZltILkJALWdoQGsjW0FrdD9EU0BbPiduKyMhW0RhcUk6VWFaPi1GSHNDcD1gITlGZlQ+XDRYRWJXRTZkRDNIJHRFKT89RTBCbEJeJm4uP3VqXksiU04lQi9LTz5VLlopUSlaMGcyNmlqMihYMj4pRUwhNTQnS15oKWUhbWQlbWg2bTdcRXBqWHJnWm80PzgjcW5SZVxjJGtJaE1tLClyOGwsKzM7b1g7XTFHT0QhRjM5VTA+YmluJ3QwM2oqW0xcZEZMLnNudSk2LTpTTXQqcFkqWSRsRW4sZD9Ea00uMUZoUWNVKVBRL08pViNMMiokSDM3ZVkrSFRdTl0yal9CNG5rZnJkaWk6dEUvTVY5RSE5J0RlK1M+YV9LZFRsYDllVXBJOyNdQD1kPjVMSl8rKiJCZCFFZkA6N3FoZURHbGxQQFxTWWkiLlFXMixgZ2goLylialY1VSc9ITsoKmMqRiZsV1dlNlc0UjkxSEl1RipNL2ZJK0lTbiYiWEZwXkhyPU0lMV5ATldgRHMwTFFZWT1RN2QhV2s3VVY5L0s7VDEvYzNTVCsmUCUxWkdDaiJnViVOUE9IKSFEMV5CZyYqMFUoSkk0aDBeb2syJEI8bjQka1YoOjo3U0hUL2wjR1owRDtIYzBELlZhT3UxZyxMJVlCcHE1cjAvPWVASlMlbEhvPGY6MGs7J2c7JE5tLkRnKTkwMiIwPl46OXEwaWtrRkxYclMtYGlEQlsvI0A/RVQqRnJedFRFOnImV0IuPz1aQiNaSSU5QE1kYG5tPFtXI0RDSCxUTmc8Ljw4dHJlcEwsViFlU2huSFIjT2BXLlg0L2ovMGJdb0VMNUFiMz1cLDc4MkYoM0dBQDBrWWlAbzdIQC1GbytyWmIjcmNudEE3JCo8UHBQSUJpTmElc3VlZDBIWTdVPk1MVjhsYWAlNV9vUV5ZZSg4cWwjR1ZqPT9CKDFLNDk7K0JWPS8vPUxfLGMjaWU7RS5NKy1AMj0iNCQqO2M4Nyk3b1BMZ2UiQExsdCEtdUs6Ni4qQyxAdSkrVCY0aE06Oi5mSkBtVj1NJj9HaiJGcHEoLV40YFRgWDRMR2EzcCQzQDZ0LjNkUWEhVT5MaDQtTVUwP3NwK1ZuKChdNzVLYEIzbzYnW15GKU8tYjY2SUQiWEVNdDViUFZfOiQiP1ZpbS9xRktCVDlVRWZfTGJHRDFlPHNjPUJTL10+UDBDP0haIylNdWQ2InAkTEdDJ0U/X190Zj5iWUhhajVnTmVuRDNzTD80LiVJXlN0IjJcZ106RzFaci4oJGQ5dF4nNXUralwuUF1wcSJpMFBbO1ldbnFxTWFvQS1DZE8wYEdJNCVFdWxYOTlEPDJDU0NjaTQ+Yl1KM05YWmcqOFQ3SzdQdGV0XG9VUSJWODVKVCxMKHNnOFZGLWlmJ2Atb1lmQGtia0FZRWUxdD02P05taCEuLS0zJ1E3MkFDaFRdPDpNPEYuJkNCTlZdIWFGLCtnWjhgY3A2UyNcT2Mpb1sqJVVZTlc1ai9jcCNzYiQ8UFhqK3RePWZJNGA2Lm1sXFg2JWU/ajlEMSNSYlpgTVFBUHVXZHBpOjdBc0hDTj50bDhtOEVWbSw/Ti5GL2E7dWArMHVSZyloY1tnKy1YLERgSGc9SSFdb2VrQCpEJEBOaTo4UVUhSmMxN2U/MUYsKC4uUmU8KjA2IjZGJzBGWT1qIiY+WUJgckMhIl8/bDVPbzBCYSJAQFZSXWpsWiVmSG91UzcvJD4iPC0wT21mJ1MvRWJgQmlyL10jKF8iUjwuX3NpUjNkdCVKXSc8ZnBUPlsrcF89SnRFWzBvaysnVGtQZ0UiOjwzUDlGLERtLnFsUDNvOnE9ZFItUzZJSlo+Mjd0UGhoc29XYnJtNiE3WkhOMlApTStiZzBqPllXNT9mK19xdTdkZiVIIUYxWCRIalFtRkdvNyxXL2d1P01MXiJoRUFSOVtAakdMQko3ZGRfJjZKIz5ERF5mQDU9ImVURi9pXSZgQFtcRzlgNycpPUBzYltKZW05MHByJnUyISNQJSUzXj9YJlFIcFgwUSVwb0tsPzheM0khQWBoRTJlSSJNPzU0OkRHYyk1dCdXI0wyKVlBWFoyJk5FYEkza29OPCYzRFVJJiZaUEtoZDgyRmhhOW5yKG1AWFJpODpaV2tKRVRDLytuc3NcU2dkcCRrSV1LT2I3PktVZDVwTidRU29mKyRiLVgyMVNbN2xINEMxUEAldUoxKGdaVnRcTC5xLXFIZWAwLUNfQ3RtR1BTNzhYQi4+M0c4WjNHXjUoay8tKFkna2U3SDpHVEhtOSY1S2l0IT5hVCFyU0EnPXMpYTBNU1I/QFRRKnIhQm8ldEVbbnFhLU5bIm9XLypkY3NkPCFhXm1pRDZoZ05XRmRvKj8wIyJEVShEaGtCUD06WFkkQjIzQWFyRV1SZ11FSixII0tvcWxjamJxWkpqdTNNOWBJYTZxOW5NRmZgSi1cZnJjLGM9RUpWclhSU3MsZ1JBVUElblZNQ1A9KWpyZGlrYENlNURBa0snWipgKmNoPzNpZkAxNkthaEhIJDw+TmFWcF4lbCxTYVNtdGduJzQ1Mk9NITBaNHBhMDlncHBPXW9ocDNASldrWEVGY1ZPQiU4Z2BKY2pvOj0raElaVj0hITFtTGZ0VTFvXEpTT2peaWIsYSpcVywvbXQwRVdqTS4yU1ddaUgoZC5fa2dJYzUoPkpeKSl0OSpGcE1gclZoJjx1KHFvXHBQSDNTYlw0TW9eOUlkNHFKLH4+ZW5kc3RyZWFtCmVuZG9iagoxNSAwIG9iago8PAovRmlsdGVyIFsgL0FTQ0lJODVEZWNvZGUgL0ZsYXRlRGVjb2RlIF0gL0xlbmd0aCA3NDYKPj4Kc3RyZWFtCkdhdFUwPyNTRl4mOkVZQmxsMzs1TFU0Z01WXmNOc1c9aiVcZWBtZiRCNyE+X2ZscUJXPHFHZGxHPUk8MFUqKklFT0NgVEpCNGYzKT9xTTRDXkk4N28yMXFBJyJ0WU0xIV1Hc0tPS0lHKSlzZGdyOm1vPSNccjtzYSg8SWIwYkItPShvSlxZNSpcMVtIKjN0OyFJL0QvVSR1RzRkPDIhYi5IMzlwVExMOls6Z19jUjZNNVhqPFpLMEcjTjxaS0gvVSldPFV1ck1RJ2pQRXVyOjEwIkQ5KSxHMy5fbDM7N2RnSCQ3YVFNM2dtPSlPVEpdZ0oxQFlDMipzRWJkalhqJFQ0LCskIWhMbmtJQ1onKVkhbmYoTW8qNFc7JDIiX21aKShAZGEnOiFuNDF0YzNoOj1AQmVGdCQ4MlJmdUJlb3RSMiFmPEwsVlgraVAjL2A9QiRTUi9wKFlUdUI9XV1VNCYrUlFRN1k6RHA3I2U4WU44IztbajhzYi1wNVAuJWZSZy9vb1NXY1s0IilhTzYqRHNeKWI3dUZfOilzbFhWby0oZ11iRkdIVTxLZVVcY0ldcCRtcCpRMUVnbFwqNytybjNlPy8iQi5LaGZVLj1mU0dgJXBqPU1vWW4hai5kO1c3JCtKOGs/ZiNdLEAoUUo2ZUEtRFhUYnJALUZMX1BObS51P0MzPlJ1NEFbLkpqRDRaUXB0PjZIKlJubFdmZEA4L0InUlAzcSNtUD5iUl9lci9fKjk1S1dOPXRsVkhqXF9aJ2ksX2VpTDRRM2JgNkVqKFUtaUtGS1RfcGhPTlxTaEopYF5EYGRncC82WW9OXiRqIUgjNCE6WEs5Yy5qJFRebTA7LTs1b0wta0NBV19WKis+SEEiXW9VJ1FaTUNTQ0BrQkhdYjteWiJUT0Q/OmkiSVg6cD1BI08uLUVkXm0uQjZNRjYlLWFNKUFGLlUkPWYwWCIkNElaKFIsJyFLPCswYV87SEYnSjpUIX4+ZW5kc3RyZWFtCmVuZG9iagp4cmVmCjAgMTYKMDAwMDAwMDAwMCA2NTUzNSBmIAowMDAwMDAwMDYxIDAwMDAwIG4gCjAwMDAwMDAxMTIgMDAwMDAgbiAKMDAwMDAwMDIxOSAwMDAwMCBuIAowMDAwMDAwMzMxIDAwMDAwIG4gCjAwMDAwMDA0NTAgMDAwMDAgbiAKMDAwMDAwMDY1NSAwMDAwMCBuIAowMDAwMDAwODYwIDAwMDAwIG4gCjAwMDAwMDEwNjUgMDAwMDAgbiAKMDAwMDAwMTI3MCAwMDAwMCBuIAowMDAwMDAxMzM5IDAwMDAwIG4gCjAwMDAwMDE2NzggMDAwMDAgbiAKMDAwMDAwMTc1NiAwMDAwMCBuIAowMDAwMDA0MjE5IDAwMDAwIG4gCjAwMDAwMDY1MjYgMDAwMDAgbiAKMDAwMDAwODk1NCAwMDAwMCBuIAp0cmFpbGVyCjw8Ci9JRCAKWzw2MDllNzJmNzdhMTM2MGY3YjRjYWUwODY2ZjU4NDZjND48NjA5ZTcyZjc3YTEzNjBmN2I0Y2FlMDg2NmY1ODQ2YzQ+XQolIFJlcG9ydExhYiBnZW5lcmF0ZWQgUERGIGRvY3VtZW50IC0tIGRpZ2VzdCAob3BlbnNvdXJjZSkKCi9JbmZvIDEwIDAgUgovUm9vdCA5IDAgUgovU2l6ZSAxNgo+PgpzdGFydHhyZWYKOTc5MQolJUVPRgo=",
    "club-odyssee-marseille.pdf": "JVBERi0xLjQKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUiAvRjMgNCAwIFIKPj4KZW5kb2JqCjIgMCBvYmoKPDwKL0Jhc2VGb250IC9IZWx2ZXRpY2EgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YxIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKMyAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYS1Cb2xkIC9FbmNvZGluZyAvV2luQW5zaUVuY29kaW5nIC9OYW1lIC9GMiAvU3VidHlwZSAvVHlwZTEgL1R5cGUgL0ZvbnQKPj4KZW5kb2JqCjQgMCBvYmoKPDwKL0Jhc2VGb250IC9IZWx2ZXRpY2EtQm9sZE9ibGlxdWUgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YzIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNSAwIG9iago8PAovQ29udGVudHMgMTIgMCBSIC9NZWRpYUJveCBbIDAgMCA1OTUuMjc1NiA4NDEuODg5OCBdIC9QYXJlbnQgMTEgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjYgMCBvYmoKPDwKL0NvbnRlbnRzIDEzIDAgUiAvTWVkaWFCb3ggWyAwIDAgNTk1LjI3NTYgODQxLjg4OTggXSAvUGFyZW50IDExIDAgUiAvUmVzb3VyY2VzIDw8Ci9Gb250IDEgMCBSIC9Qcm9jU2V0IFsgL1BERiAvVGV4dCAvSW1hZ2VCIC9JbWFnZUMgL0ltYWdlSSBdCj4+IC9Sb3RhdGUgMCAvVHJhbnMgPDwKCj4+IAogIC9UeXBlIC9QYWdlCj4+CmVuZG9iago3IDAgb2JqCjw8Ci9Db250ZW50cyAxNCAwIFIgL01lZGlhQm94IFsgMCAwIDU5NS4yNzU2IDg0MS44ODk4IF0gL1BhcmVudCAxMSAwIFIgL1Jlc291cmNlcyA8PAovRm9udCAxIDAgUiAvUHJvY1NldCBbIC9QREYgL1RleHQgL0ltYWdlQiAvSW1hZ2VDIC9JbWFnZUkgXQo+PiAvUm90YXRlIDAgL1RyYW5zIDw8Cgo+PiAKICAvVHlwZSAvUGFnZQo+PgplbmRvYmoKOCAwIG9iago8PAovQ29udGVudHMgMTUgMCBSIC9NZWRpYUJveCBbIDAgMCA1OTUuMjc1NiA4NDEuODg5OCBdIC9QYXJlbnQgMTEgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjkgMCBvYmoKPDwKL1BhZ2VNb2RlIC9Vc2VOb25lIC9QYWdlcyAxMSAwIFIgL1R5cGUgL0NhdGFsb2cKPj4KZW5kb2JqCjEwIDAgb2JqCjw8Ci9BdXRob3IgKENsdWIgT2R5c3NcMzUxZSBcKGZyYW5jaGlzZSBmaWN0aXZlXCkpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA5MDIxNzMyMDgrMDEnMDAnKSAvQ3JlYXRvciAoXCh1bnNwZWNpZmllZFwpKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA5MDIxNzMyMDgrMDEnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAoXCh1bnNwZWNpZmllZFwpKSAvVGl0bGUgKENsdWIgT2R5c3NcMzUxZSBNYXJzZWlsbGUgLSBHdWlkZSBwcmF0aXF1ZSAyMDI2LTIwMjcpIC9UcmFwcGVkIC9GYWxzZQo+PgplbmRvYmoKMTEgMCBvYmoKPDwKL0NvdW50IDQgL0tpZHMgWyA1IDAgUiA2IDAgUiA3IDAgUiA4IDAgUiBdIC9UeXBlIC9QYWdlcwo+PgplbmRvYmoKMTIgMCBvYmoKPDwKL0ZpbHRlciBbIC9BU0NJSTg1RGVjb2RlIC9GbGF0ZURlY29kZSBdIC9MZW5ndGggMjMwMgo+PgpzdHJlYW0KR2F0bTxnTiklPCZxMExVb1VEJ1pgO25nMkgmRVxVMl8iMkI+SykkXmI3R01SLDZKJF0jI01zXl5TY0V0JzE3VlgsVUpkSyFKQStXZSMrLkIuRGVrM3MoPFkqcV0xK1o7YlYwdF0qJCRxcTMjb25raTsqZWUlPCtPXFBEO1ExZUQzMlg+XVowPylNb1dbai9ITk5wZ0AhKmZSczAkJWlPMSZLPUNWMk9ER1RlUztZTmlyMVIucVtaLS1WZEsmcUgyaS9wTURiPyozTmwuU0Jfb3VRXV9bQStPMG5lTSphViRiUi9WYDE5NXVRMzhSMD1aciZZRiswamZYbUBiVUQpSy1eRFBtZjFcJz5aOiNoQSJIa2BhSFE5IXJuR2QlLSZbVC9aYVJCNltpckMlYEpaLnVOYCNRQF4oRmEuIydmW1IzX2NlWUcxQllQPzphRWVNPmItODJUMkxzYiMkYVZkQicsL2M4TGtxYERLXGdUdVIkWTlpaDw/NzkwNWQ+c20+P1JXMXJsKFglVlRoMHUyKEgkby1dKFs/ZzteOFEsTEUyUytPMF1ALllvTFtoWjdRPTF0VTRaQiYzdVA0YENXPXRrZjhhbDY9SztrTW9bZWlYRUNSRl9lK18/cVRQQWpkbi1MJlVNRnVaYilNYlkuVVVYN0I7LjkvbmNhbzJDMiIwcyZXRCM7OEZrdTdaTnAwJSIyVkwxUSZJZElMYFtNYTI+QCQ1TV0pazpWLm1fOGBqY1VsIWF1JSVSZEc4NkRSXTk2KE46UCtMS0EvNm5UUFcoXmAyVHBrbldnJCdlPDE8Xz5BSyM7UTtePEBFb1lAdSEpYj07cyZRWy1vTmthJVFvWStNOmZgayYhcTwnYTFxNkoyKHVVMV5HYz11ZjxDTEomKWZUaUpkQWFFXlg+Sz1GWUlHdExbOWE3V1xlLFhJZlstWGhFUEglXyZOTz9eQ3BTLEhuS1hiQ0s+Ul85TlZaZ0FeV3RKM3IpWGNqbzRWXCIrI3FEI14+Z2FfRDAsOU1TIl5XX1VAaiZuKSs3QlwoRWErbVQ0OVY5MUhLPEFhbi9DXHI8Ql5mVVgxSlAhP2RZPS9ZV0pKVFVdbS4jO0l0XVFvMS8iVSgpNiElYikqP0JnYG1MVnMsVkgtVGokVVcoQnBxJTp1ISJnLGk1YFoxdVNFazZhQE0iI101WF8xOzxVYHVtQnVuNU1FVFAiKENpRW9xbnUrV3ArWXIvUU5AUTVSTj8kNypcQiEuZnQlXD40cjgxTFgxJy80KHVqJ0k6cSlyNGohKGFqOFxyRTNjKDJnbGRxJGJnJCZcZE0pZSJcMUNrPUc8QFlRNj0+aS46R2NsVyM2cV1KWDdoVG5SWTFXTVJUN3AyMWlhOlRvI0QzSD8jS1xEV3JLQ1UtZlBDX0Q7PlkyQiI8bChxSExuQVtFMWNtYTVEamspcXQ3PlBLR107XSxoImNNQUNgSFcqcFYrOj8rRjdbNWlySC1jcCdMZzkvVGRjRjFKJFNiZzhLRU5FXldqSkhJcVFZJmUjVVtkP15FcERtZGZRLj5DZjRlPTZwJVU/bG9NTysmXS9iaktiXCVGPm5NI2RwXy41RGljLkI2cmw+XD5vSjliLFgrYWYkY2lZQUhYbzQjJT5ESUdlUEk6Si1uWTZybTcic1FhWDQ4U0w8QClAYGRMWVlwdG9WPGMvWFJdal84NGdpT04zJjVCITdQbDFdMiJWLltAdCIlOUdlNVlPcGtHJlwyIV0zSmg6JFQ+TlUmOGh1PTZTIVJ1PSE8MEJCWV1mZi4zJlUiODQxUnVsJVVkYDU6Q10zPnE8IzQxUV1HZiQmUT8nTzhWUDc7JzUmNS1TOGdMUnQ8ZFNhZTlrdWtuOmBpXzxmak9DJUtoZSlkUVQ+UWJiaz1hPXFqbkBgVC5VO0I8RFU5JXA4M2BVPXBoP0JIP0JuUzRicXJ0ZDN0MzpRO2BwSkIzOWV0XSVtQTVpV2FvNzNYXmkqI3JdUCgsM08qKGFuIm5tclM/bGkyOjhVTGMuVHJuNldTbiQ+JXBIL0YiRTM7U1EncXFLRTNoRyhVZStJWjQ5ZilKOj5MRyR0YmhIJGIsb3UoZ21DXSRlVz9xPzVfJzZyJClHbWYiLjQlNHMpYDckLCowUCM2az1eLkJNRmUhV1doYVYuaVRdXCcuTTBKaXUuPVxEOEI4ZVI4MTwzVzBjPTtBRGBxL2MhYjZnYElbQ2VncCo7Im1aNWdMc3FHO2hyUVMmWSVMSGRFKHJKVz89NC02IkBoLjRAYSpsSGJucS4kP1NrdSFtVFAmdShgK1tEbFBYUVQxYlcwL0hYOjArIVU2cTtNN1xRbHBXXk8lJUs3bjMmakdZM2cxSXAzNmJcKkhVU2o5UlNtJk5cP1NpKSUrR288TkcqcmNOKzxBUTFpdXVYJ0RVY2Ykb0pGL2doW0wkXWdwJFpePzZYUXVNK0xFcyRNaXV0ajVvOjNfdDomXywkcShnPFMvS0JpNlh1USV1WzlMRz5mQlBCU0ZcXmBUcDpiV3BpVldpXztgIlFsZDUpNU1kbDkiYiovUzlkZSI4MmkpbzVTTyxPU3AtNDEhLTs3QiRnTTovJShGQk0qYSkrPTlhO0I4IlZda1ZCSyNlMT9yJVM8SChmXk9wWj9vR2dCJk5wSEArZjNtZlBJR2k3QU46MjM6T2FBcEtiOEJrODNocVkvNCE0ayxjSEM2ZHNGOSNmcWI0NlUjYlQjWUE6JHJuKU47JE0kNm5EPkg1WilVNj1NVWsodUdrcGdxRjVtW3E8blI8VWkzdWFpMC9jWVgjYkMpYF5DaDsvUVkyM1hQRXFTXiJwcEhDcExOXTtHaWxLTT8vQWcxcih1JCo7RzdNY3I0NG1ZIzAmNi9ablpSZSUvP0kiY21ZU0Q9L0Ikby88aSooWmEnbCg5cVRSZ1c5aEEyWkZrazo8OkZjcEVDc3Unc21wSFpbMysxUydxLj9fU1M+bEQrUT4sTjRHN3BuY09EPi05VWEhRnFpLDhCMnBGOSpDZ2lhUldUQFY8KFYwRHFFUEsiLEl+PmVuZHN0cmVhbQplbmRvYmoKMTMgMCBvYmoKPDwKL0ZpbHRlciBbIC9BU0NJSTg1RGVjb2RlIC9GbGF0ZURlY29kZSBdIC9MZW5ndGggMjEzNQo+PgpzdHJlYW0KR2F1SExnTiklLCY6TWwrbHE+QW83bztpTFBjYVIsUVg1aVs+O29EQGR1SWQjJkAuQ1wiOllvV1AtPT9CSmk/LShVYTJMa0I0UWklWWAjaSNjUm9icktSWV8rYW9eTXMnIys1dEwzTmRDJ3VbSEslJGp1YV90KTRSVVVWMSVUOF9ya2FUQjQuUCZcLiUwNXF1SE5xZyNxSFw8b2ZoNGJfN0U8PTwtWDtURTA+KltaJElyZFxHaWxvU0xTJ207VDhpYE9sW0VsMy5DQmw8ZS5kOkg+VixoLjpSPVwhNWhwP2o3UWpoOEVhTm03aGIjL3MybkFfSXBxZ3ItOkw/MUwhSUxHPFI6Tj8vNW4+LEVwUU85T21bMlJiIVRMdDxjPj1XOWAsMWk2Y2wkYCcqO1Ykc0pJUUw9alBSXlklPFVUVEBndEJdUDBmZE8+bm0pMjk8bFBtckopUzczaGQ6W2NKZE5NKFVPJVAzXWc7XEZmMyY2ZUotJDBTJDNfV1thdU09b05CTi08OyViOEgqMSRWdEFYXz9tUVlVVTBFREMuIWxxM11qQ19DIzhfbCRPODxdXkkiVGkyO1goQTxNLjo6YWA9cCo3JCM3YzVgK1JWUlxnQiIxQ0hbYkZ0ajNGIz1rYio0czM8TzlfJStmLzckJW1DSUptSm8yIllUV0dQRlhUbkRHYCVfLkQ0X3RrSiZCJExqSU89XWFXIlNMOERZO1ojUGR0Ol0jSD5xMkhRQEouVy1bWVwvUCxNVVJDVTIkcktvZysjIW1zak8zM0okJXNCc2YydG4uTDRSMyVHYERYWj09QFcjTD1hZmA7K19oZDY/WWJAXGEmRCw2OS03PilxI1skPCotNXNfPUJHMFZUMXU8Jj1ZcElBXDlXNVlNM2IyaGc4VzRPSCpPO1AyazhaLzA1MzFAQE4wWSN1QCY4Wmx0PC5cRVA5VWdkM28+ZWwhOS9MT11VdVFwc21YXF5tLWFVUUVtISYwREAxaj5zXTkyQipyTiVUXlpjSGFMa1tiXkUrTDtBOj02WlMwL1tucj8mLlJlSzN1S1M6JHM2TWZhJVFmKk8tJCFDRC48PytFNGpbPl1AaU4qbWhZLjEjRi84dW1yTW1daSs9Yit1WEY0T05zXV1KXDRXIzYrP15pP29UJz4rcGEuVGExYTlZK1ZwX0NtMDE3SSI8TVh1WFw1KVQyJEJYLyhTYSs9cUxwUyc7UkBFKmAxNz1kVGpoPlF1cF5FbyY+IzdRIzFjXm1uckwxSlhKJ2tXQi9qQjpmb0BPby11VTMyWCxGb3NIIVk8UDl0OjRCITBSODg8T1MuQV00Kk4iWklUM0l0UmJXSTBuZlo0LCx0SlgkRmo3Vixib18hS2BqbldoPC4tbjtfQmVxRlsxNVZUIl9hYzhgWW4vKTRYWmxYY04tMU8ka1pGLWxGRFVHTzdSIScuayI0cGxkPys4QG1XY0JdL1JwV0s0OVJUbThHTV8wUTFRS0hYZWpOLzE4JDYoWy1NdGdFWHQ4PicsTHNDbkZuPDM0OSQkZzlRLmJqXGhrTWVmYV0pJSg3WkUnbTYiTl1iPVBKczEscWRAK1pVNXRfLyQiYjlePCZgSWNFUSNHaycvUVJTZ09ZSERGNXM0KUtYNCE3XDNXSjI2REpRQEIjUUg5aSVNKionTWIyPGJsJmAybUcqSjdxJlQjUj4mbXFyPihQIlxoaVkrOG1NP2JyNFhrUT84W2cmQUhdXCNWLjpmZyYrMXRkK2diTVZDcmpVJnIxQ3NLcTEjKDo/MjY4KTVrO0tSYGkhZFw3T2RIbURNKyIlaXJWOWxcaWtPWTxuTTFyRTgiczZOdEstT19YZ2E7MThvIihbNUF0KksvbDVfX01sR1AibitJLmptRGthRCRraHJETV1iUFJtXSsiL1NdMnVucFpASFlKZ2FvMDZJPiZlK3BzW11XYUZcS1wmKF1UbmktSyxrbGEiOWsuP1dcTHJONVNcTG1DKyZQJkg6MyRSITwiKyFXW282RFckRVFvYUhsWlVFUlFJZFw6RSJASStFK3NcIWVdYUUlNSpaZyZeImF0KzYxWE9fIm5gYlZwaGouaDZqUTpXWyQkanE6MlAtK2RRUWdxWCEnczw8JUBQai46VUVZbDVYJWguVU9sZ20+Nl9FZDljPyYkWUswbylDQi9TdSE8PmxPI0VhZ0tGal42LlJlaTFGL2o2U3UiYFc9NCM4WTFBUnIhcnNvaDdVVihyKUt1SD5wOzM2RlEqN0RCJzNTTi5lUWBjKy1ZX09kKmxzSUl0R0I8Y3VOMiNAQ0glOWFNNzwmO04lZi1LR1whTFhDNDt1XyY/JDImMiEyXk9AUGg5dGwnVkZrKlRiW2xsTF0tWmFHV1RkOjhRYHRbKiswN2FxLC8jMmFPZl1jMWhpUWhPIjNCTkU5TjhyUjJIVkpdOUghO0knVEZVQiJMUjFYYG10V1w9SFM2KHAkSWpfU0hLczA5Si06ZCNsRUMsanV0KiRDa0pZWS5sWTdBNkY4Q1ArUWQ/LSJFOmArYCJGRmlUQj9fVF0iaV9UPC0qOTdqPkFcYWxaXS8+SWprIW1XdW4iP2pLISFLaFhFYFQ2c0lyViYmSEA4ZEFxXDdLXnRZVTwwVk9vRzpoRVxuRSNvYScmNyhbWilIKyQrclJAMzBiJC9BQGJtW185S3NJTl0wN3A8OEFQKDVJSSxtJUhGPF9kUDNSKEtrRUM1VT5FWippO25CKCY7Wlk5Ymk4RTdOcGI+PVtsaWpoX1FOLSZYODlGQDQ6WXFLVUdLKGw4V1lOKlVxKiUmOVtTSnFuZC9PWWREKCNAMlVffj5lbmRzdHJlYW0KZW5kb2JqCjE0IDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoIDI0MDcKPj4Kc3RyZWFtCkdhdFU0Z04pJTwmcTBMVW9VRm5uTUJUY25hUStNMGU5XDEoYiIzOiNvPDs1QCkkMiUmQ11LKzNLbFFoUHBOLW9HSyFaNEk4Wj9OZ1xGZkVgNTNJJEsxN0kvTFBRLVEwa2tjcltvcmo9W1NAMjRYL1omcDBeOEgyMDJvISlLXGdMU1RGTEI0TCk7WEtJQnNxL11uL2NKbmpMMERhSypeLUFGS3E+Z0JqN0wmMV1HVHEzVGRIKGNNQGlJUlRgXj1VL0wiUG5yTkJSVk9CaHUuZFozbklOQz5qMmAnXz81LF8qQG46aWxvV09CZkJsJ0ktSlZHW2w8cygjRTtxODZaNVs6cyk4J15gZUJnNCNqYjVOcFpeNF9ZX3JFUEI0cHBaZ0EvV05rdWQubXU0UjA6WmEvWW1sK1VWU19uSGxaPycnQ2MnXHI3amp1XC0kUUZMbm47SDtDRTVwNGknQF5Gam1IYTBmZkU+Sy8pJ3RsXClpVHU2QEpzN2YuR1MmK2JhWllFM1lMLGpVRURBYy5aaWRodEdhJl0lMGYrTyk5aHBuWDpHSW9AYjQsZHF1Im87KzInU28hWFtCQ1MvTz0oPD43PlhcPkY1U0kpOjppJi4pQUNZQWtuIlJVZ11HYD9HOzJYOjh0SVU4dU9DNmVhZ2ctPjVBb0AuTzxEOj5IK2E8I2NgWU0ucyhJJkAzPSxoQDJvYidsdWhYLzYiZkpsV1wjVyVpNy9KbD9vOmVgXSdRWmNsOCIwSGxaWmIwPSE5PVEpY2hMazg+QjssKmglMHM/K2ROclktMml0KWtuUGkiK2RzNl5AVCwvJlVdWWw6a0FAOSdUNCw0ZyQ4aikoUEFnQDU/V3FJQnBYXT04YFZjY2RrTTNUWmNXIzBqN0A8NXBfM1QvJC8xM21CPjBXQExrQWZqJS5WUTRlVTIzalwhIUA7KStGZmY+Z2dlOVpQakFVXSxLZEltST1qUy9uMSRiRDFITUpsV1MhIi1TTmQ/TnUrSFBPNDFoQDwwOWhaSm9TPzMmU1tpNDpwP1QicWEhRyN0KUUncyddVyFFNi1VUm8tIzZMLWcsOkFxQ1I+YDZxYmJWQlQ/amtRInRBZiNObyssVTZAP1RFSScuLDZdRGdGXjBtPUBDKyc2VTxdRTgsSUdLQUlBUmFFZWQtS2tKQyUlS2NiYGwoZGIrRWJualR0IXJMJ1NJXzNtXitHJk4idTIlTi5uS2VLYEVUVixTKTM0aldIVl4/RSFJWi5gLEFGQ11dZkZwQkRsRyxQTTg7JjIsQHBMTztsKlNbLF0jJWlQaFYmRyhQKFY+KUFRLVdxXnFAJ1tPck1kVydbO0dTKyc5LDFyXkw4dS4/STw1UFk4W3BwcCVzXkU9LWloU2liYVNEKCxiYWtsJCxiIm1ETCxPXE5zLi1Obl9WNTVQWnJKRXJSXE9cOD0yTCFjVmcxKjpxRilfXiZyZTdSNkpMUDlmJzY1UmZxJ1k4UWkvQTRgKlZcLjhDPzt0cGI9b1kuYmUuOkAwJihhUUI9N0AkWSZuQHFOQC8qWiEhc0soOkZYITM2IkdYTW5OYzo2KS5AQ2g3ISkpP1AmQmpYUTtOOzIqK1NHREJQczk7Ny9EN2ozb2hlZ2grX1I4LDByZD40M0BQRmFLKXVIVyFPNidpTWE9JEhAISwjbkBvbzkhVFBmY1AvOUlXKmk3LC1fcHImU1FQSXNwOU0pMVgpbyQpU3RnXW0vQ2ZQPUFhXypPNGknVDdKWidbYExPQmRfI0koOituLz5gXilUVipbaz1CVFtTaGA3MC1xPUo2QTlfV0tKbCU+azdZJD9KKSI6XEBmZENVZlcvPE9gIjInPmZFRm84MyFGIzhuTj9vJFNCKi9oVyxpb0dXXWRPIlgkc1YxSk1qNG02Jj42R2o1WkxhW2c2YiMiVjshPiwvIlchdU44YEhMSUhMQShyXE5cUCxyYzUyQz0vbSslV1Z0QkhDUzgyNXVdbCssdEolVUt1WmNVJXVKXlNrKSdqUEFkXz9QPk1iMzl1KW5lOWlrYmAiKTRnQlhBa0sxMWxpbSdQW21DZkIxUj9wWTdhIk4sb0omJTsmQjI3ZTBfNC9JRzRtVGxOM2tBaXE/Vy1bbksrPDAvQlBPOVBbJkkwJVxeT2ZTWnE7dS4vWmUlOSI+SyFKUllaVjJYSUttSk4jUVVtWjpMUW8kLUczRjYjaEFhN21gamNwPkcyMiNpTzhfSl4zc2ZLPyJaOFMyYS9nREI1MzRKb15LYlJlOEdQLVAqUF85I11PJltwJi9tSFVYPEVMKF1PQFIxJlNFVVZ1ck09VlVhJ2ZDc09jNCoyUTckREwtSUNrXFk8J1VRPWMhSjFoaDYxNXIxJjRQXVxgLWJbIUMmUTRLWi08ODJCOmJjY0FpKls9NGNvZERrOUs5TDwudTA0TGg7OEI6RGpVN0c/VGpFVGE0WFVJNmclYkRqdSs8TlhwZnQ8PShIQF1rWjhgcT84ZlA/cSxEbF5mT1oxaUVKJEElVEIzLHNiQGQ8ZicpQC5uZEpKS2tWaiVyIjF0cjdNa1ZbSFxtcCsibEt1Lik2YFtTbkwjOGsxKDVHXXItMEMxWmFNYHBAXXJWTFglcmg5R0FkLUVCQVJPL0tSSkFxYnQ8IyJfO0NvY2xELltOYCo1RnIzXDgpcC5NWCVHLEEhdWhiMDspLmZfL1g1OCkwN15nakdAJWMyYk9OPUFdcjojM0hALSZQRGlmKjQ2cDopMz5HTkxdIV8yPEUpNzBubDE1MDtPKG5jVW81U1s0PDMtLERDOTxxIWo0WmxfXDhzPC9wQjciSShxXl1UIjw2U0MwXGFWP2hJdVw/L087OXRjcWQ7SEk2PTciWUhvWUMkNzpUKGRQOWk8WW9wTUIhY1xGVVhQPVNjYDpAXiooPC1fM3E4aWdHcGhwUlNGKFUmdHJVb2JfUVY4cVktcT9rcFkpbEU4MUola1hYWicvLl9vV1NEcVg3Ij9bQV5ybnUxKy1VLXFmQU5XUUcmMTdyJWBzKi0kOiRsQChOK0FqWCReT01JblVePytQPURLc1UsR0BuXHFPPmQ6aUlcRHFvMjxzZDxfIWJSNWNQdD5jbWVfdV1wcUZmV2RuRF47L0RIdGNBNVNLUkBcXi8wV28mWFNgLiRAR0BSPml1L0xsXFJQXGAmSVQ5UTBuQDFqQ0xMfj5lbmRzdHJlYW0KZW5kb2JqCjE1IDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoIDg1MAo+PgpzdHJlYW0KR2F0VXA+QXI0XCY7QiQ1LyddKkA4ZlpTaiFgKWhFLTtsL0RVbCQ4Y1pDIT90WklkQi4mR2Mna2gnSjpUVWReMCIvOFdZVmNAImdBQWs7YT9fUydjOVJmXHEjKTZ0NU5LbCErYjs5V3VgUzM8S1VdPyJ1YE4yUUouX3BwLy5aS1pWVkgyTydEWltKdEIuTitwblw6QEFpXVpaRzZyXzQrTztXb0cnJW8vUk8yNShYXCcsSDcpQCw+Yy03T1NBbUcjKmBmVkJyNCluRHQ9QEZfa2txVmlRMFAnP2dfbmxhNEtmY1dwNVpJYnAwcHI4YkBlNk9TYUtfMnMiOT0rXzJ0Q29DP103UiUqc1lJOFQ5KitcIkJiKlRFcm9VXSUxOG11OzNJIVpNRD9YQFw5LUViaShjKC9xPiNdNDsxXFllR3EyQT51LGYuaCRham9hK0YhaEQwRTw5LVFbVFVNMyM9Mkl0Jm4pdWExVDwyUiMjRGxac0EqXlhTZWAwIVtLSzYsdGkpLjlmJzFFSmAjQXNaV20sOVtdSGwxUHUnQlpYTjUxSUFFYytJLk8sR2E6UHMlZC9mKTE3NUdAODxQJV8ucGZEV0ZaJlNDSWhJVmNGaTxcRlIwcV5jVG9dWVYuMCQsUk10RloiTGo2bmtuakRZTUVLLyhrTVY0PVBdNFhgW2hdVG8kZTs4OEsyU0VAIkg5aUohUVY2ZDoqREJfSVk+ZW9bVjg5ODZxJ1hmTW8hdFFVRSUyQWtYa0Q3ZkBAMGNJX1Q3JlIqW1JZQSpxTEhsVUBZVG50JzklKW8ycDNdQTlxPFdJTSU3WCguIV9bdUtIc0dgWS9mLyxwOnJxN0BtVlZdUldrRDdDMTtCMUZoZ19DVV5FcGRWPm8jUnBtJ3I8K1tLY3RIVHNyYjdmN1ZGZ2VdRiM3OUhGWjp1cV5AQStmRGJNakJaaHNfTFokaCswY1dMJXRibiVgI05jaSJyLydaWEw4XTFSPUMjdCknJGdyZ0pFTjVWNzwoazVRNUA1Jy88VlYlMSYmJy11XGJyXWUsR0NBOiQmWzw9WD1VLipyREBvO1pedC5tTCVbdFlBT25GNi5rRD9vLjxfR0koMC5DO2VbTjIxcClES29kL09+PmVuZHN0cmVhbQplbmRvYmoKeHJlZgowIDE2CjAwMDAwMDAwMDAgNjU1MzUgZiAKMDAwMDAwMDA2MSAwMDAwMCBuIAowMDAwMDAwMTEyIDAwMDAwIG4gCjAwMDAwMDAyMTkgMDAwMDAgbiAKMDAwMDAwMDMzMSAwMDAwMCBuIAowMDAwMDAwNDUwIDAwMDAwIG4gCjAwMDAwMDA2NTUgMDAwMDAgbiAKMDAwMDAwMDg2MCAwMDAwMCBuIAowMDAwMDAxMDY1IDAwMDAwIG4gCjAwMDAwMDEyNzAgMDAwMDAgbiAKMDAwMDAwMTMzOSAwMDAwMCBuIAowMDAwMDAxNjgzIDAwMDAwIG4gCjAwMDAwMDE3NjEgMDAwMDAgbiAKMDAwMDAwNDE1NSAwMDAwMCBuIAowMDAwMDA2MzgyIDAwMDAwIG4gCjAwMDAwMDg4ODEgMDAwMDAgbiAKdHJhaWxlcgo8PAovSUQgCls8MjhjY2U5OGVmZWJjODFlZGY5MDFkNTRhNzUyM2Q3ZWE+PDI4Y2NlOThlZmViYzgxZWRmOTAxZDU0YTc1MjNkN2VhPl0KJSBSZXBvcnRMYWIgZ2VuZXJhdGVkIFBERiBkb2N1bWVudCAtLSBkaWdlc3QgKG9wZW5zb3VyY2UpCgovSW5mbyAxMCAwIFIKL1Jvb3QgOSAwIFIKL1NpemUgMTYKPj4Kc3RhcnR4cmVmCjk4MjIKJSVFT0YK",
    "club-odyssee-paris.pdf": "JVBERi0xLjQKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUiAvRjMgNCAwIFIKPj4KZW5kb2JqCjIgMCBvYmoKPDwKL0Jhc2VGb250IC9IZWx2ZXRpY2EgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YxIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKMyAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYS1Cb2xkIC9FbmNvZGluZyAvV2luQW5zaUVuY29kaW5nIC9OYW1lIC9GMiAvU3VidHlwZSAvVHlwZTEgL1R5cGUgL0ZvbnQKPj4KZW5kb2JqCjQgMCBvYmoKPDwKL0Jhc2VGb250IC9IZWx2ZXRpY2EtQm9sZE9ibGlxdWUgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YzIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNSAwIG9iago8PAovQ29udGVudHMgMTIgMCBSIC9NZWRpYUJveCBbIDAgMCA1OTUuMjc1NiA4NDEuODg5OCBdIC9QYXJlbnQgMTEgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjYgMCBvYmoKPDwKL0NvbnRlbnRzIDEzIDAgUiAvTWVkaWFCb3ggWyAwIDAgNTk1LjI3NTYgODQxLjg4OTggXSAvUGFyZW50IDExIDAgUiAvUmVzb3VyY2VzIDw8Ci9Gb250IDEgMCBSIC9Qcm9jU2V0IFsgL1BERiAvVGV4dCAvSW1hZ2VCIC9JbWFnZUMgL0ltYWdlSSBdCj4+IC9Sb3RhdGUgMCAvVHJhbnMgPDwKCj4+IAogIC9UeXBlIC9QYWdlCj4+CmVuZG9iago3IDAgb2JqCjw8Ci9Db250ZW50cyAxNCAwIFIgL01lZGlhQm94IFsgMCAwIDU5NS4yNzU2IDg0MS44ODk4IF0gL1BhcmVudCAxMSAwIFIgL1Jlc291cmNlcyA8PAovRm9udCAxIDAgUiAvUHJvY1NldCBbIC9QREYgL1RleHQgL0ltYWdlQiAvSW1hZ2VDIC9JbWFnZUkgXQo+PiAvUm90YXRlIDAgL1RyYW5zIDw8Cgo+PiAKICAvVHlwZSAvUGFnZQo+PgplbmRvYmoKOCAwIG9iago8PAovQ29udGVudHMgMTUgMCBSIC9NZWRpYUJveCBbIDAgMCA1OTUuMjc1NiA4NDEuODg5OCBdIC9QYXJlbnQgMTEgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjkgMCBvYmoKPDwKL1BhZ2VNb2RlIC9Vc2VOb25lIC9QYWdlcyAxMSAwIFIgL1R5cGUgL0NhdGFsb2cKPj4KZW5kb2JqCjEwIDAgb2JqCjw8Ci9BdXRob3IgKENsdWIgT2R5c3NcMzUxZSBcKGZyYW5jaGlzZSBmaWN0aXZlXCkpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA5MDIxNzMyMDgrMDEnMDAnKSAvQ3JlYXRvciAoXCh1bnNwZWNpZmllZFwpKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA5MDIxNzMyMDgrMDEnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAoXCh1bnNwZWNpZmllZFwpKSAvVGl0bGUgKENsdWIgT2R5c3NcMzUxZSBQYXJpcyAtIEd1aWRlIHByYXRpcXVlIDIwMjYtMjAyNykgL1RyYXBwZWQgL0ZhbHNlCj4+CmVuZG9iagoxMSAwIG9iago8PAovQ291bnQgNCAvS2lkcyBbIDUgMCBSIDYgMCBSIDcgMCBSIDggMCBSIF0gL1R5cGUgL1BhZ2VzCj4+CmVuZG9iagoxMiAwIG9iago8PAovRmlsdGVyIFsgL0FTQ0lJODVEZWNvZGUgL0ZsYXRlRGVjb2RlIF0gL0xlbmd0aCAyMzk2Cj4+CnN0cmVhbQpHYXRVNGdRTDtOJVhsWilpNj1EaERRYlwkZ1JibEcuXlBLLFdsXy1ORGdBJCZnODdyRydONEtJSiVsNV1EWFIvKD5yMSQxWnMrZUFpOEVNbkVfSSdvQGVpNT1NO0leYm4lWmpWLWFZczNdTmQiJXBJbFpRbiVRTW9dOko3dVovLWheUiVAPlNNZzJramlaKzcvPWElaTBZUmg9N0BBXF5vS1RfLSMtNC0wIixGRFJkW29oUExvXkpQZnBbaFxoaD44a2kuYj01YDNUS0E7bnI1KCpyPWVTRFsvMWUqP0N1Z0JzMiEjc0RQYypFPjZsQUxIZytcRT05W1NrZ0AzLjhgO0U6PnMoRy0wV0hQOXI/WCF1W1BQJDEiSF9PJGVpVUAscktxY20uUUNDQGkhQ1JYQl88cCs2MnIlJXFwS2xbW2hKUjhLXSc8XmA7dDU1W1QiZDU8P0Y1RUFXaGctJmhZKFFoPy88LzBvLyZjSzdpYlhJTWhicT5EZUctPERsQCYnRl9TMnJZPClSazo+K3VFNDkySjpNZm9QRDk0JDUwMCkoQkdPNFRVSDQkYFloPXAvRm4iR21ZZFhJVWZKMU9WPDU5WyYpUFpWLl1ITj9SMT8vQFhOWFJZdDVBWFclRFBNJz1GYSxqU3U5VkJELjhYZSdTJDYrLG1VRm50Q1IpTC1IXFktT15CTUduN3M9aix0K1NDQnU/YDg6JUE/WCRIZS1cLy9IYz08QDtQVCs3cSE/aC8pKyZqXVFJJzIxQSRbTmpJJ1JVYiglLUxmb0VCQ3U+bT9SRmNQLUdIODYuP01XLHFIS3RZK2lwTXB1KXArQSZvTVdZOltlc2g0LF0xbSpwYCgmM2VBTD0tZnU3PiErXUZtTGtZZzlpKCdnalhsYSJaPydiY0deKzBOOGZfMCFFIWYiYzMtQSg8PDlsWGdvL15bPjkwQlVVIUlTRS8iWipoL0RMXDdXRWM3Jmo1YT11Qjw0NVguRk1TLFtSVyIxX1Q/NVwiSmlTKDFqXjs9Ql5pUGEyRiwsOkQ0UyNOJVxYPyklblwvalpyUSQuWFwxMjk6JSZqb1s8UTg+QW9MPzcsR0VoN2BYX1IrciswJV8nUW5OaFdXci5naEJzYy09WlAvP09rWzAkbzFtJHIsMCRTWU5eY1JiRXEmJmFASWY5YWQ8SF9oLExFWl84LzZFJls9blg+NCJcJD85YFhtRitXZWcyUlRlW0RVNi44XWxlOkdCNUNVaSVzL19KPkdTPiMhUiRAZD5ENW5YNWpDS0ROTSIvWChrS1lhdW5qZDZWS1hUPC9BNTZhJGFfZHBTRWZQUmFxN3Rlb15ANCNTJFlmVzdvbyszKzM7KDlgTkBoRDdkbi47dDJScWtrbFE1Xj1iUEpBYHMzIUZhPzJzMnQnO1tmbmhaPjpEdTBCazdPI2VjSCpxc1BvZCNiRyglJGFYU3RBa1pqPlY3YXJTblxUS1JgUDppNmljPVgnXT5Rb3EzLHNGVT9nST4pcEZZRXBwU0lDZUNfWFNcWnJGRCchT2pmVVplVjdHUkJiKmAhQjxZSG88SF9oQV5kaiQtX3RoKScpKENgaj5KNkFoZmNXa2pYREpPLy8xKUk4YVhJL2tfYUtJJXVNW3VjWWhoMVxFQk47KjpealUyJWtTN0FZR0InQm44ITBJOSVhNm1JJik3RkYxTlRzMCs0QjdaL1tIU0w8LmlJWytwIyg4MlJuWnVnMSdIRHBuL1xaM0I+Ok1EVU9ebkwhJmZdM2IjRi9iK2BxNF4rYk9xdSFYI1NCaDltSzhFZEpIPlE8RkBZUVpvKEdiLy9VcDByP1wzZU85PSVDOlcnNVkoMTxhYVQrS2FZdWUhKVh1QlkoYUskQ0hPZ0liNEZHTyduXVRRPjduczZqZUdbKFY8WUxqazBcQGM/X3AxPyY1ImI3Jm91MjU2YTBtN09eZjlUVl5obyozRVwkUF9NNGRWcTgoODxVXjpVRSlLLEUyKC8kRTRCIWprJ3E6YCVFPU5XbEJacGxvJlk9JVdCJ0dRZS47cVNzUm4+cSFcPWtbKmUmKE1pLGklbWo/KShDcSR0c1JqR1haUks1MTc9LzAhRihXMyUpIWpVJzJgXE48bjYsVDxXYEZQMidZQClDVWFdYHEsPDhUOF8qWGwuXThibDBVIW1kUGtucE5gVl1cLDoyaEhuJztHdClRczVfR0UkZj41XXI8IkVYKD1HRlxcciJMajQmMWhWY1cqRmZkVUgkSjBEW0EzcGRyJ0tlOWhxU1hWKT5CXUQ8a0RtVyQpQFwwSE8oNUg9WUkkNUw5Z2o5JF9OUitvSj5sQSRQIkhnXVU+KlJ0ZCtwaS88PU9aYSpKPkk7MnN1ZFU1X0dBM1g5MmtcQT11QTA0cmFuPFcqKyduOi5GKC5hdWMxUzRobi5VbFZjbGk8aFgxLzlCQyQxUGJuYlZgUGNkMnJSK2VJPT5yWj5TKV8+Tic7ZElEKzI9LWZyTF0/UkFsZz9GbmBSNUpdbGRYLlZnLWBiQzU2RCdMc1VKSUNOWjtkRSUxWytQdW9XK0FacldjU1ozLChLcmVfPCI3RCxrKlZHW2NcJm5bSEo/YjplOW1BTT04OV9dYGEycVsrWS48VDNsI3NURWVNXDtra1RgUG1NYTxabDxwPVFBL1M7XiNPXXRuNGRHKHJeaWMqMXVMMXViKD5WRUhjbyVKMnNIVz0/TG4qLC07NFFzaTtePkElOVBSRVhjNjgrKFU/TSFIbjBOZyhtYzo/PHRgWlktQyooIWJGTENnVkhIbEEmLCNLX0k8KjNPbyhjR1AiNl8uOnVtMF9KMTg5TmtUXllJPDc4VDIsW05RYTg6THJbWWJCTEVgb0ZFOmlWWHJHTiloPz9LdCwnN0ctYEJJQlVKXUxpW3JqKE1WO2lzZVFiZSFyJWpPOzg1QzxIXlEuRF8ySiklLmJLXlomcSVrWmhmbDxuOEdeIiokZiFmXio/M1YxJ19CSypjOmRiXG5PWEYnRiZzRUE3WTxcOFAhPi49SSYmJkAhVzhxYUIiQlEsbEI+X1s9MXNjMzJpZFMzYztlWFE1SU5zWGMtbyg5LitaTVJfTCtFZ3U6JCxwcm8vOzVKR29yKFdcLikqQUNrWjR1YVdyUE5kTkszcCopPzQrOF1FWEMhPDJoWWVPU2BZaT1+PmVuZHN0cmVhbQplbmRvYmoKMTMgMCBvYmoKPDwKL0ZpbHRlciBbIC9BU0NJSTg1RGVjb2RlIC9GbGF0ZURlY29kZSBdIC9MZW5ndGggMjI0NQo+PgpzdHJlYW0KR2F0VTRnTiklLCY6TzpTbHE+QW9MSl5ZLkBPbGthUEI+ZyZSQ11aT1I8SkswSldQUFAsXFZkK2oqbEUjKFZFO0k2bi5BRkNlSCUoaCokLFltI2hzNnBfMzEpUi9wUjxpMFc9RkxnbGAnKzt0XFkwWTwnYFpdZzAuVypWS0AjRXJqYWA9TEdmK0laJyZcLjxNNUkoTlhUaSFFOC1ra1QwT0wwPFpiJS1dUj04blhgRF4pTiotYDV0V0BnXmA3Rz1lSToxOT0yYmokWmwwMWUiOz5bPjMtSkUkSztWR1VBcz5wQiFBRmtvTWolL1N1W2VjaE5UTT0vdSdAPUAuaz1JNUpwWD8+OmVWSF1JPmpgXEg0NSY4bzw5XlRRNi42USdUWHE3Qi4jLGQ9KElsVGg3c2lxIU1aJCwuRjNwLXJST2xpUlRvaU5RPypfUGtWYi1ZVjlZTm4vRUZdbDIrW3QhW3VDcl5UJyoiZW5hbWdJMyIkJyRIVVJqUzEsIkBXTjkpY0EqZGJkI2ZrPVBSX1w4M1pxUGcuailRRmUqOkBPY3E1QysySSQhUkcuOitUKzsvMy8pK0VWWHRkREc8bjcscDIqYGRaIVBSZ2NxUj0mV3JKZil1V0tjRi1yNC51VkdePkY4W0E7Y1VrXy5ccS9ZVnIjSj1OKkYpaiYvR0NUZkc4Jl8mNVAjLjg9T0JjZ2N0OkAoTzdDVTYxUk0uSGNeVVpvRUthYlQhaWcjXWhSJjJLOExoMyQqRzNHYlwvPWMvamtrJmFWUGNyLjhvRC5nJyM5MDp0THIrJmlCY0ZbST00bV83PENcNSJyL1YwLVphbV8jIT0lUWA0YGFkUWVaXWw4QGdjLEg8N3Vjc2g4P0JaJWBcZVZNMFUlMmw+V2BwN0xaNzFXLTAqaS1dPmBtTzlwSEdYPzZJTCtfLjwoXS0tMlVSNz0/PTs5SCY7cHFcQlRcYUstbztfcWY9TUI/ZSExZ2o1VWVJQmYpZWNfV2YnSlpnOjVMSm4rRjs1KSpoYjw3bSxLb18pMnJxcygrZE0rR2dvX1FQUUcvWDRDRUlZWkRyPjM6aGVRNipwJyJhbz8tc0cpdTlQNS9ZaDVgJFxjalElXCwrW0BAUTZkXlY9XGhxSWImU3NULSgkXUlMMWo4RytcSzEoSXRrXUErS0csbUk9OHVKLCc2a3JtNCYjaCVuM3JAaTFSNmZqRW1bI0xtY0Z1LTQoKy80JWJSbDVhJ2Q1dFpNQzEuLlBmKFtZPzI0TzRTaHRSVjE5NCg0MXROO1tlQlBaMD0/X00lQlsnWm1CMSYnRUMmU0suIUEsN0YnVUwnSj0+TmxWMUsmLzhdSlYsP2Rxc3NFZk0jPyU7N2EtZEI9KkVWcUhpQC4hSVdpbTVzKzwoSyJEW0JjYV5BNHJVRUZGNys2LylQcVRacj8/STtIUystRVVeKV83WzVmP2wyLzBeWlBbPGAlJS9oWEg0LXMuLGdcOkVcTG1VYkxLUjcuJCdRbEYoLkk4XVg/PGQ9YlI8aVpSUSsqPj1yZysmJi1NMS1lQ2hmRFhtayMkLGMqU1FGcTEibj1tMypgYCNbR2BHSUgmYVI3cypYQzU2L0VgaCguVXA3LXAwXzFHaFQmMFNwTGZeIkVsLExTT2tkWjVCRFMwZSNIKUxibHNiXThTcFxjKjo0WS9XOExtYVtrZ0p0XWIlYmk3TWViJlFOIzF1YWVROFhnLEo6TUwpJnRjWzJMKGZkVFd1JUcwJzhMNzlPKiNeblUqUl1RKSktcWlrSG04NyFWbVYwYk1hJiM6PWomOUFoVU9SNSVMYjMtKXUhRD8rY1lcY0hTZFA5WyJMNVZiIzB1YF9BR185cFQ5bVdYKkEoSUFwcUhnJ3RhIUg/TzY6cGIrSzkhaW9KdD0tKiRmcmlqZjxuMGpHMCdmQm0iPD5lODRgL2xTTz9rT0dhOFgxSVYnNStXJC85aTE0Ikxcam5aT2tHNyhISUQtSztBWj5nJEFMV0hZZCdISGcnLyFtI1U6b2k6b2lbWnVxS2QjclVdVlw8KFJnMkNoR0EiQ1onV1ttLF9uLV1qKU4jIjNpYitKcCI2cylWSy07W1NWWHBybFNVRkQ6NnBLTFA1b11oWjk3XlNicls9ajUkYTRVYWx1VyZHbT9HOHBBN2BcWmZBck47Zj5QPEslWEYnMGYkVV4+X2xgbiNtR21XUjBpNW9HKyQwLEFEKTw1LmoiXFhkJV1RKUFWOzhXbGdCXVUyN2tXNTxPV3JqRyJNXUQpQGVsRD0sKT0majJBMGpRMTh0OjUlKEY6OzJjcXBRWVRyR2dVS0QvTShGVT5GRlFjMW0jSWYlOEVzLVtdcWdXNTc6VEJZVWw9JnVUWC9kLSkrVyokLCg4cTFpLUMzYG9IRjloN0wxXW5ITlxXXko4YC0paCc2JjRUM1kmOj5pSixvXmUvc3FAK2RdQ1tgOyQ2IyooVihkb2ZjJm1QLHMhb2ZIYVtLUW11c2hyczAnPFgrLmB0Oiw2ZlgxST5kNkVxJGJhJGtLQ04laXRcZ0cwaW4mSWRWSFI/SSgmKiRQWjNcJVZzI1IuPlFaPltmZDhdWF5CPVZLIU1PKzFscHFtYS9fLVtpZWIndU8xMFIraCNTOkxiR0BjT1RsPS8+WUkvSjdsbUdmQz1oPVIjKGk7aDFeJ3AiQmpBaT81bTlCNGhCOWstZFtZQmhybkwtUmAzcWI4c1hqMWxOPUswISxEV2krLEgiUTQmMitRI09pTnBDS2hSclJ0a1ROb2ReZ3RUTD5gP0ZJTmRrUGxPMHM3R1BvZk5kaU5UX10kc11EX08pckguYS5pbSo4bzZYMW9zOS1KZFguO1JOUzRCIjRaSzNLTDhJLVE5KTM8cCxkaD1CPWdjb05IPTRhakJjTjopW0dxIkxUPXRCaj84T3VASzNcSVtkbDs/alxsUHA2RSU5ZmszZ2YvXGdHT0B0bmZNcD87WFh+PmVuZHN0cmVhbQplbmRvYmoKMTQgMCBvYmoKPDwKL0ZpbHRlciBbIC9BU0NJSTg1RGVjb2RlIC9GbGF0ZURlY29kZSBdIC9MZW5ndGggMjU0Mgo+PgpzdHJlYW0KR2F0VTRhYDg/QiZcW0tcbmNrLEwkKT1iZmNsPSRzWV5EVHNTLWFMOl1HLnBiNV1NKksvYiUrb24hQDItJFJqL2o2YyJqVDFvYlNFcShmOXQ+MlVjcHMsMz8wVmFwVVRoZkhyLG9RWyRWNF5aW3VCRFY4cWlRSSE9U19QalxnWUhuKjVCYE5BXj4mJ3AlcHEhTkNPPnNaJDJHXzlERGc9dUBpXSw1cGh1SkAxMmklLj1KdEN0WzljW0JGQGsxYkI6aTRvXU1oalZkbCxeSD9DXmcoRkknWFtdKCZgKFdeOVcmO0x0RV4uKlw4R0xWZF9OZCs1WHNbLkU/PjUkZigrKTxXbEJxP009b1ZUMXBLOXIhKiMlO2tCJzpyZXEhNkAsYk1Cc0suU2NUInA/PHEzQF9QTixUQS41VnErNUZYRmxXYWIzbFsmUSpSayhsNm48ZC0nbWdhaVlwM25XInBLVGhyVkljb2JlQFssdW80NnRGS2RjJU80ZS9nUT1xMUl0YDMmWjxcVkdbbFw1SUBqN14kPy1YVj9uO0FpOmBoSzRXTGolMHNubitESkVxP0UjKjRgaXNLUV9NODtcOygiQ2orZVAvVTkwJVNfWy8jIk8/aG1DXE1kQSIkU0NLV3VZS0IrRUZEQVZvYElEM3Q4b188QDkuOy9KV3FbYmJkR0wyKl5NUSJPPylbX14tYzYqLUtDU05KMVtuIUxhPXNLI0dAaEtULixOWm45M0FvZWsyLGNwVTIhZileXzpJSiFLZGIhTj8hIUZcO09mSS4mQTYkMXFPVUg8LzszXmBjRShPNGhaZ2xHXzlaPS4/YGdKX0koZj4mYTwsYHMqLUt1SFtWNlNBaWA1VlBtSTxrNmRmVyljZTJTO2BoYnIoKy4xbWJcPGcjOSIoaWFIKzxPZlZXRjwwQkRhQlJbXDY5NkZLci5LR0pwbyQwSDVhLWgiZHFFLjUuN0QvWiJIOlszTGw8UyRbbkhbbGdOKzxBaEg6WkE1QTFlKUlxQUYnYTBmRSE3PTBKKzpDZFhlYV4zMmdqdGo7ODs1XktxOCYyNDZOX0IzXWdUUV86U2llVS1rOiVdazhia3RWRj4jKj4oJDcnLTxLKlZjNEJsSFgqaGk3RFY8Y3QqV21eZFZEaDxVJWBUbmpBPCpbSTRLMForYVw3ZUtITDZEdGkkI1lNamk4NFdzJ2VUTFA+aTwzKDBWM2VQOltQa084KipxVyZPVDVQYSVTc2ZuZXI3W1xla19UJi5kSzEpNEhGYlJGKihnYC9IYT4sVHJNSldZWSM0JE0rKnMqX3M3U1wnJnRuU1ItRy5AI2xoT1Ylbzh0JmFWLDU2c0oxXSJnc2VVZ2s/Uj9UUy9zU05lLkBGLDVmI05sV19hQywzRGkhKz9oIlcpQkAmUm9tVTIwYCM4T2tgPz5pTlYvL0NtaTA1XHBwSmxtQjprWCdzPj1yQCYvYjw9OmxkNSJQJSFidFdlbU4mZkxPK2MlZjVhSnE7RFk3Yk5bWFhRL1s9TF0tJl9nLWJmTWc3Vzk0Lj5YYEAnTjZKXFZOV2syQTskM2BNM3MlbyFva2UvSFM6ISdpYSNkV1VMcT9kW15vLCskI01GbT9UNm4qYzQia1ciWSQkVkg8PVhBOiVFOEZIY2xkYWc4MFc+JSRobFMvVklFQiciMDkySTdXNT4saEdZbHM6RWNuQywiY0JMKGI2VmNVbDdXYSVQcXBYXSE1QWk1NjkwLT9ra0tocDdpXSI3Ny86ZiNKanM3bCxibyVFTjVDMDpNRGowY1JHLkVhPjpMSTE1O2RCYU8qNm0lMztObTFHYyZIYkFuO0M1bHMyOkcqPDBNMk8rQUYyYElKITQnOm5tXGo0bzkvU2YtZ3RPXXNXMi4wK0JVKiJZbUtDTVlnPnJUazppT2lhJVpyTFNzJFklUlpMOkIjJFxDVz1CbWNAWmtgcSwiUVA8TmU9NFI8YidJUEY1OS4oTj5QUVVQOyQqQWhTZSxKPmhWV0hKP2tEJEUhPzhvbEw7KUE3IT41XC1BX3NUNzRsMEVDWUE7aiokZDpoLF84RDhkWzdxTkcyLypsIy8sXzFBX3EiNXU3UDpKMVsta11PXGxQOjUmdEtTTER0OFRTIzNeOW46VTJtRGgyMWBYYz5UTVxkISlAO0gydFU+Y2wkSVdEc20kMCtHPzpuTSo+Vld1WT1iYT9qdDMubVVONjhKYEpGYjdcTGpDKjlILkNDZlNTY2FVbiNCSC1Cbik5LUNZNUpbTTFNOyMnRy50bVcqL2NgYmEvSWFPQnQuc102SnNJI0hZRiFgPmhEXG5IaGJYYkY4Rlh1TyJzK0NsUEJORylGNHFENztiZVI5WzhjV0hFRkJdLU05IkNPNkksYFtWK11II00iZ0IxSTs2OU9NR2IkITtNLT00TEloUyMjb2ktQClVPWJnbTtEMCkoOyJLbkk2IUtiXlJNcis3OSYxVVxrOi1kMjdzbmFdTWMsOT9dMjFDW11DI3FSWysna0xbL1xuOW8vSmAqMW0nNFx0aXA+UnM6Rzg4Sk5PS0FjdCo2YFpgTCg4I1dNYFFIZ1wvTVZkNyNGNDdGTkNQJSVgc3VoYk1DTEFBOkw2SUBcb1RCOyQndWBzXmA9Ujk1JzElYjc7U0trNzYjZlFXS0dSIkgxKUZGU3RQOHVLRG1kVnJmY2xINj5CKTNtQlpvMmN1UURIPzw8YVBZSC5bRjAqOGozZ1hbRUdBIjhSUyg8MGZRSkRhVFhjRXJmLDNzcWxfZl8rKWo5QjBjODhoS1EzUGZgYldmaitZXCtNK1hRTDlRYTZeXVQuSEFtRmBFYmpEcGk2alQ8dUAoTzBCb1gqRFQidFxgXmgua19UMWBNLmJENyZGTU40XCMpWypmIiosOUQsaV4zKGZVRV9RcGdsaXJbSiUrQThjaFA/dDo3I0lZXWlURltkUSZdYyJDNSRUIjQuRUJnOWhbMmRmK0RFLkV0P2guL1VMTmc3IVhURytJJmJDaiU+Z05kVkZwXWNQPkZHcV8mXlNNPi9qP1psJXNtMjBvZygmPCsuPzpEcCRlMC9DbGtOb0pqYDJGYGsmYiMkNDRjRjxQYW8yKFkwbmVmLm5mdWpvTyh1QF9YUFA5JypRMW1RXm1GKDxiRz5xQ0RqUit1LElnLGNgXEMjZz1rWzVYbWF0RTFETkhJJFdIKGNaXzQoQmZROzQxJVJbdT4yR15oMSglaSNMUHFyRm8lVDw/KGRta2FpUkQlSy43VDtJTiteSD4sUmwlPVdRcG4uQTVYN2tpM0BgNUsxIjo/JWNcaGFJdT9zI2NMLywoYD1HRT5OckdgbFVFNzEiJXFJKWJuRi0lSydFTTdqJV48UjNYV2N+PmVuZHN0cmVhbQplbmRvYmoKMTUgMCBvYmoKPDwKL0ZpbHRlciBbIC9BU0NJSTg1RGVjb2RlIC9GbGF0ZURlY29kZSBdIC9MZW5ndGggMTE2Ngo+PgpzdHJlYW0KR2F0VXJoZiRzdCZCRV0mPTU3IWJsOlRpPVVKRDBCT0g5ZVg7JyhobWldPl1CLSpdOStXN1pVR1NYbFk2ZTNUcTFkdWxdKTQ1XCo5NCVEUCU1RVlyRylXOEw3aEwjcVlMOU9kMihfPVdqUilAMi5pcjI5TzFULy4sRj5mV3FcL25vZGAsJUlaNCpqVlVVQFU1VEsqYSlERWtwZ1U0cTBkbDRMLlI4JD8wZGpTNGRQJj01ImRpJXApazUjWjwsRD84KSt0TEMxJlE+WnRYQTdNRl8iSGIqbWxpOCNuY0QiQlxsXCFRVTdSSzVQRSQxO1tmJktdYmdZTi5KPDhWVy9URTEtWCRmNVUmW2YhLVE7b1BjWDlMZlsmQjxFcWtJRSVATFFjc1laIVAydDM2SjwmMnBTIlFuSlUsT3RrSyVQZHBrVWAsbiNJVyoyWVRHNnNocyM9bEREP0dJWS40SHFnZzsyLV9OaDhIKU5PWXVTK1QiRWlLTm4tTyNUYlgiMEhSM1shMSQ8QVlnLGhPP1JAM1s2RCpWMXEtLCRyYThHYkQ4PXRAPmVnSldEJFo/T08sXzBNJGluY0BsbTJaIjdSW0BQMHJcblRvRlwpXTksOEl0VGghZkRqJ2UzUDY7cFY6UyEwNFclIVxqIm00QEAxIWRQST4lOkdqKkQjL04qRGwnVjU7NEBYbGZzJ1lWUDdCTTYsV2xiby5BdHEhclJEXCI4KUtzZypOKlEuVy1paGgpSUJHIVNRKXNzXGlIJkI6VUJmVFBHREAoYXIxbVRnTjtbO2tXQ3E+X1MiS1tEKEBRLnJRPUNWRz1iQ3QqXzspPElBbDZMPyxzTlA6Q2grSVk9S0BobEBcRTNGKzpRS1JlTlouOSInbDU6NTVcTFlFRHFAMCdGJWAkdGg4Yz5YKUZtazpsbSVNYT1cNE0uVEY6K3REPDtgJlI0WkRrdDgkX0swZFZhOUBkOUAlPTlLRWdKJTwpWUM+VDtLTCstMlhTLTgtRjlLQ3MpMi89Vi5nOHNkXzdtP1tYR2Bxbmcnb0wqJ203WUMqLjdjOkxlaTc2cjslV05FaXFpYiRPS1pXU2QpcVQ+USFJVSZzSklcOTJRcWdjQHVAcTJDdHUscFlyRSxNckFSZCRaNiI9RENyUik2QSFhMWtgSz9BM18kQ0dnIWAlIzZJKi07Qz1WMmRUJW9PPG5uISwqRDpaMnFfMmlfSE5vTz9uZSl1Y0Q0U2haNDQ+bjZQVGdzQjhjN0wvSGE3PzhoJ1Zab0kuMDw/S21oVENJaWoiZFJwZzJyLGJcTjMmKHRaYGRjR2djQFsxNUBWMD9IN0lVbXA0TT1LPSZNOjMrYmEvSjJALGxiRkonNVdra05VcVVnZSlwK2IzUGpdRUluMk00K0xOTjkvUiFNNGRqalElVjQpWWs9aDMrMFVsLmhZaC9DSWdkXElWKklhOyRiNz9TdG1SJWtyJmFUM3NmbGokLlFOZVYsYV1lUCNEOm5ZXzpEYG07Qj1mIWdPPTQ1cXMqRCIuJU9hUDpaMClicFUnIlFwfj5lbmRzdHJlYW0KZW5kb2JqCnhyZWYKMCAxNgowMDAwMDAwMDAwIDY1NTM1IGYgCjAwMDAwMDAwNjEgMDAwMDAgbiAKMDAwMDAwMDExMiAwMDAwMCBuIAowMDAwMDAwMjE5IDAwMDAwIG4gCjAwMDAwMDAzMzEgMDAwMDAgbiAKMDAwMDAwMDQ1MCAwMDAwMCBuIAowMDAwMDAwNjU1IDAwMDAwIG4gCjAwMDAwMDA4NjAgMDAwMDAgbiAKMDAwMDAwMTA2NSAwMDAwMCBuIAowMDAwMDAxMjcwIDAwMDAwIG4gCjAwMDAwMDEzMzkgMDAwMDAgbiAKMDAwMDAwMTY3OSAwMDAwMCBuIAowMDAwMDAxNzU3IDAwMDAwIG4gCjAwMDAwMDQyNDUgMDAwMDAgbiAKMDAwMDAwNjU4MiAwMDAwMCBuIAowMDAwMDA5MjE2IDAwMDAwIG4gCnRyYWlsZXIKPDwKL0lEIApbPDI2ZWRiMGI2ZDQ3NTdiNjU3MWY0YzQ5YmE5ODBhNjc2PjwyNmVkYjBiNmQ0NzU3YjY1NzFmNGM0OWJhOTgwYTY3Nj5dCiUgUmVwb3J0TGFiIGdlbmVyYXRlZCBQREYgZG9jdW1lbnQgLS0gZGlnZXN0IChvcGVuc291cmNlKQoKL0luZm8gMTAgMCBSCi9Sb290IDkgMCBSCi9TaXplIDE2Cj4+CnN0YXJ0eHJlZgoxMDQ3NAolJUVPRgo=",
    "club-odyssee-toulouse.pdf": "JVBERi0xLjQKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUiAvRjMgNCAwIFIKPj4KZW5kb2JqCjIgMCBvYmoKPDwKL0Jhc2VGb250IC9IZWx2ZXRpY2EgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YxIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKMyAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYS1Cb2xkIC9FbmNvZGluZyAvV2luQW5zaUVuY29kaW5nIC9OYW1lIC9GMiAvU3VidHlwZSAvVHlwZTEgL1R5cGUgL0ZvbnQKPj4KZW5kb2JqCjQgMCBvYmoKPDwKL0Jhc2VGb250IC9IZWx2ZXRpY2EtQm9sZE9ibGlxdWUgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YzIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNSAwIG9iago8PAovQ29udGVudHMgMTIgMCBSIC9NZWRpYUJveCBbIDAgMCA1OTUuMjc1NiA4NDEuODg5OCBdIC9QYXJlbnQgMTEgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjYgMCBvYmoKPDwKL0NvbnRlbnRzIDEzIDAgUiAvTWVkaWFCb3ggWyAwIDAgNTk1LjI3NTYgODQxLjg4OTggXSAvUGFyZW50IDExIDAgUiAvUmVzb3VyY2VzIDw8Ci9Gb250IDEgMCBSIC9Qcm9jU2V0IFsgL1BERiAvVGV4dCAvSW1hZ2VCIC9JbWFnZUMgL0ltYWdlSSBdCj4+IC9Sb3RhdGUgMCAvVHJhbnMgPDwKCj4+IAogIC9UeXBlIC9QYWdlCj4+CmVuZG9iago3IDAgb2JqCjw8Ci9Db250ZW50cyAxNCAwIFIgL01lZGlhQm94IFsgMCAwIDU5NS4yNzU2IDg0MS44ODk4IF0gL1BhcmVudCAxMSAwIFIgL1Jlc291cmNlcyA8PAovRm9udCAxIDAgUiAvUHJvY1NldCBbIC9QREYgL1RleHQgL0ltYWdlQiAvSW1hZ2VDIC9JbWFnZUkgXQo+PiAvUm90YXRlIDAgL1RyYW5zIDw8Cgo+PiAKICAvVHlwZSAvUGFnZQo+PgplbmRvYmoKOCAwIG9iago8PAovQ29udGVudHMgMTUgMCBSIC9NZWRpYUJveCBbIDAgMCA1OTUuMjc1NiA4NDEuODg5OCBdIC9QYXJlbnQgMTEgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjkgMCBvYmoKPDwKL1BhZ2VNb2RlIC9Vc2VOb25lIC9QYWdlcyAxMSAwIFIgL1R5cGUgL0NhdGFsb2cKPj4KZW5kb2JqCjEwIDAgb2JqCjw8Ci9BdXRob3IgKENsdWIgT2R5c3NcMzUxZSBcKGZyYW5jaGlzZSBmaWN0aXZlXCkpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA5MDIxNzMyMDgrMDEnMDAnKSAvQ3JlYXRvciAoXCh1bnNwZWNpZmllZFwpKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA5MDIxNzMyMDgrMDEnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAoXCh1bnNwZWNpZmllZFwpKSAvVGl0bGUgKENsdWIgT2R5c3NcMzUxZSBUb3Vsb3VzZSAtIEd1aWRlIHByYXRpcXVlIDIwMjYtMjAyNykgL1RyYXBwZWQgL0ZhbHNlCj4+CmVuZG9iagoxMSAwIG9iago8PAovQ291bnQgNCAvS2lkcyBbIDUgMCBSIDYgMCBSIDcgMCBSIDggMCBSIF0gL1R5cGUgL1BhZ2VzCj4+CmVuZG9iagoxMiAwIG9iago8PAovRmlsdGVyIFsgL0FTQ0lJODVEZWNvZGUgL0ZsYXRlRGVjb2RlIF0gL0xlbmd0aCAyNDAyCj4+CnN0cmVhbQpHYXRtPGdOKT5hJlVnOlRrYSprXy45W21vTWwiKVI6MWVNcT03JTlcajFJVyZCWWJpWCVSPWNRcyQhPiJdWkRDJmIsJnB0T3BFblVxal1rK0EzciMiJnBfX09VN18vT0xYLThRLklATDg8ZjdeW1RCT2MnX21eLkpeTixNajJTNCd0ZCxYdEJPbUZ0Zm0pLnBlMCJeOVdZYE9mNUBBXFhtUGdZcC5WQ0EhLTJpST9AW0dCaj01LFw6PUFUc2Y+b0FANFxcbS1WT3Ayc0hmU18qTT1YcjptTV9hZF1AYjM/MmdeU0NgOG09Lm5CazlmLUVoc2tkMGE1cTQkLEdHTFpZTUVHOWYib2FlW2YhUnEiP2IqMVQwJjZQQzdLJWBbblwwKkZSdCRMN3NeZ1sqVDNrWy5AOE1JbnNRUGxmMDVwPyckJ01aYDhML0kvdUZoaihEZ24yXTtwM2Alc15wOWlyKW9qczJlciRkTUBrRE5cW0YrXGk2LFJxSSxHa0Y8dHEhMC1TTDonMlo6Qi1ZX0k5TTU2SC4tY14hWEQtUC9qKERmNms7NXBMNWIlPTAoRltVQHJkXU9tNWtzYTloZDtCNk0qYCssSFpWX1M7ZWddNTEuY2trSih1ciFVcU9VK2YnZTVFO11EdCgzQXQuTChpdHFANmM+YmBCQC5jRDVVXzAjODQ9aUcqVz1fWExmMygtX24kMT9vUTZVPzxAcTNkZV84YlJ0RFlgZWdXNEpfXWxoIz4xNjhAaTJqQjRmPSgtI2smRy8nMlldQTFvLzowby5XMEhSc0RgS0JXWiwiWyFqQTQtNiRrbC1KaDw9QVw/a1pfWzwzPEAtaS9vIkklIz46VSZeKC4rS0I1NXNLa2ltITtgaHVhdTprS2RyWzVvRm8pZVlgRUVmRywzVU1YZFY3bjckLmkzVl49TCsrXjE7MixMZzBMXWQ8OWNbVlckckphRCVIRVVvVi9pRkJeOVM3dUVeWVYpUSRgYF9GJk9nU1JfOzVaQ0Bgc0BCJCtHWjdPLzxXQEs6PUNSJ2lkJylwc19GamY6LGFZS0JKRGlFcnRrZjdVTy1pXUAlaWJOOEQjZ1YkKyxhW2VVal1wZUltcGtsRychTVAzQ2pOZV5RVFlIZlUzNlllUmFhPnE/VXNpSFNtUjxwSDNeX21gXWFhRVBKWCI8O1RTTFtxOjdGImo6SSJxcFpiZjU7TDQiamwhUy1dZy4jRHE3b2tUZC5vTU8pRGFKOzB0a2ohXFdbcnA7I2BbLG5DOGxRYG5IJC0tSFVTXk1uTk4zSDlFaHJ0Q0puTi1YOF09SDBbQGcubj9jUkNAMy9jKldXTjd1TjNARypVblM7OjVrSWIlZGA4c05cKyRaKnQ6VWxkJzY0PjFubiQwRl5AL1k+RVVnJ1I2V2F0WTBQOydyVSltNjNkOSVlY1RMbGEtJmQmXWgtKz4tcDxqXiE6WmlMSTJVakU5JTEoVT0wZ28iU0RdKHVOLUtNb3UnZDUrT1l0WVJlK09JYm5oUV4zXFdrcS1WJ2lWU140QF4sM1dYWjMpL0xhXzRlZWRzKUJtRDJ1ZkxmP1giKitsPU1GPzwtcm1tQkwjXCojW2oyTUBwSk0mWnVnNllEJ1QpMGJCRnApMFNjbDJaSGVoKy0vKTRbUHU8cVFHQ29gamU5anQjOkVlKSFWPzFgTCdFNm5yIWNuNS9UOEdlOWtcLWQmRU80PEleNWQ3WUtPc3Q0KFNKTDMjc2ROR1t0X2NvWDU+PiNSKmgqNiZNQltmbTY/bCU4ZVtjZjRXKkkpInFJZC9hZCldS0YoKlo7Szs1SFRMKyRySC5vYV0/JTRURi5LZ1tyLFptIlw6M2YxY0JxV0s6JmEkK2JncGEqIm4zSyQ1VVtQUC0yNSRwODhbSHBnXmxpbW9FLjFpLks+N1dscDNpT2omZW9VUEFbOlRYXUNvWERdSEQxLzdwOEM+ZEc6ZD9KMU5MRUh0ZyE9RnI0JSxMSyooMmNmUEZWIV1FXj4vSkJtSE42YypXX1Y3VD1uLE0zUHMzbmRtMUhIMEBDbyYsNVhtIlM0L1hkLFduYlI1RVZfU0A2ITY/ZEgwbEFvRkh1Wl9RPnI9UDMpL25YbzohUUQzLWdHbHBpZWFWcycsXS4/WFt0RGpZVzhaSCFGW0VOJEVmXlg7LjZEI0JQJipEU3EhZTNnKFA4W04hSW5gMTY/IWRiWD9cQ1JBQGw2KlIwTG5MUHNSPGJbK1YiKTw6ZWJcMj8xLyEybV9NQ01GWl81Xl0oclIyMiEkaiw+K1hfITRpOCRkXEhubGlQImtwYURFXUs2c0s9MS9VX2QlJHI7Uis1LldiNlw1OmtmMllmaUopNDJYbUw+RTpnZCdQTkQwUl1pTjIwak8sbEk8O1BPM0xwSiVKMTtWb2B0Zz1wTHFLTClCYz8hUjJmXzRQYmteVUctbD0lT0NDW3FeUmYsLGAtZmNTJlVrUSlvM1lcQjs0JlBXZC5AY0BCUSkiQmQ+NWgwaEZcWl9VXyNGUlxob0UwS0A7ZSw0XnJgSTdbLm5ZPyNVaSlHZSpVNWVSOzNeLFJEczYqP3IkIihvbzxfayJgViQnRUxFSisnREdARFxRXm4tXkhgTF0qV1o/YywkKSI0R1dUaSdMY2pAZyNCV2tQYTRZJGJzO0Y+NDVmZjtqNyMmVCVFZ2BfSlUvYk0hMVgsYFVYK2I8JlBZU0xcako3bi1zTiVVdDxaN2ViVyMpcmplLiIjQSc/XGpbQDYhWyIzK2ZSRyVbOjJvaF4hMGs7XXM4TmBrYSEmQkxTYi1OOVdLSjhxJDNFb1MpIXFLZW1mJmZzST9tQlBybF5qTDVNOkxKSFtbZl5PU2VFZnI8YWFLZl8pOWVWYzg8SDQ8PWFnQEojV2wpPVIxVjJAPkw7U2s7cXFqYiVvZGZuYSY1Ym0nNCklWT1ESFBtSj1SO1tmNCw0akJYWjI5a0htKDBwa111KlFIajFfSU91VV9yM2ZWSzA1J1pEM2ZhTF1OIStiMF1DUStlRkIuME5aL0pYVjpHMFkyPjwvLmNvOzY6MlMhT1daLyphWyRrKiQmakZMNUsqXUFIMG4+cU04NTxmOUBBTTcjNisvWC9Ub0xhcj1lQ0BEXTlpYXNuVCQzJWhbZjBdaE4jTzRDQHRkKU5MLkFIOlUoQDgubDxkQ1N+PmVuZHN0cmVhbQplbmRvYmoKMTMgMCBvYmoKPDwKL0ZpbHRlciBbIC9BU0NJSTg1RGVjb2RlIC9GbGF0ZURlY29kZSBdIC9MZW5ndGggMjIyNgo+PgpzdHJlYW0KR2F0VTREL1VAUCUwIypqXyhTMShfPWQ2b1FGUWhBJ0okYmhnVWJyVjBIYWBgNCNwMD00J2VZR2ZDbWhiK24qQkZpbitgbSEoTGNyLGteWE03NUNcQkojUmctImp0O0BJWC4sVDNoalROZGBIR3BNdGNrdWluO1NfWywhX10lOzU6WzRKZkNTZmVBdUNqdUEuWDckZGYlMUl0JmBVI1U1U1YrJmVROV9hKydhdGQ8TF9kbzMyZGhzYjpVWFMiYFY4YSNBTSpnaiE/KXNIbjhNMGMkNkM3QSxoV21GcC5ua0BPU3UuKTAnRSUtPGpcPSokLjo7Qz0pMzZwUj84KSthQU1TbTRQWEZySSdsPFBhPighJF9jYGlFJUlkanAmWihSRm8najIyYk5VOzlUWkdTOVEzXkIyQVBxMSwtQzpZQlwnIWBsN2JHJG8nWi1mb0M0RCkhQEFLWTUtMzdxJyxjbV8wJ0BNLGBacShWL2EiMktFSERccEw6cGBSWkJmZz9tU0lTYVZhXkM/ZC4yK3QkWUhKTysvLm5wbmEoQjIxPV5eJStSLXR0PiEjUjZeJj1nV1I0PDl1OD1IMDgyMClJOWptVFFgIzduViY7cVpbMmUoOy5ZSWtuL0guPVh0MlshMk8iOzUqRW10VVxIbUNFQ1cyZExVSGs8Pio8clc3Q2RkTWluMl9BdTAjcVtQJzJAIy1GQkkiNklRLlAvbyo9KmhmTk05aigmPi9HRUQ1bTBRXydeR2poU21PXDpMKTskaWpOZVdHJWV1VHNBSDRsN2E/Z2VlcHAnbSJgSzYyK3U3KGBwTEZdXjJDL2E5cHB1NlFIREJeRW8lU0g7NWJfaCJxQ1hoSVxCZ0YoLDs4YEQmRmsnUzpoQFdALl4qaCJGSTRQVkowYllSZHU9ZzpEKF04LE83NmUwQVAiNmd1XCxuXT1aVTQhPV1ZaFZXQUE7MFQ2KlVuSU9zZT1iJUNRalJwW2s2QDZQIyU6YWlaQlhXXE5vWFAmSGY6K0pjXDE2JDhSP0EkPy88QmtCYkwoLlVSMT4rQztGTCdXOm9TbCxBTF00ZlIkXFlHLUtrTmkuXFJ0LThkWVFKbzkzOj5GYmtjWVg3M1g6SlFDSkBFWTVhTTJoVCwjWygvO1dmMTMmajNdaHEnNDUnUEpXajk8Y2tJXHJAWC0mb1E4ISYnUTpTMTJHInRgT1coVGlVY1dQJnUrXnU7cCE4QjptSXBlKS49UFc0UlhZRTIvNyhBYnM8RDhxO2Q8IkhRTlooVicsI25ZMGIuOHIvM3MkVGtQMi1JOVY9OGxMaE8qcFJHTDc8RTNtTWFLXE5wTSYhPGklSztDa0lUWWk3RVona0lpRCxbcVAlOWFnUyE9REcwZD9YRj86SWJBMl9nTS5NUGQwIXUqKykzQEdyKCY7PWM1YmNuSFlUZ00hJHVvJjVENkhfJkVZSVAiUEY0S0NCKlFMRiE2QGxkZWkiRzRLJiM+XDIvPWctOGsoVT1BaDZxQ2VlMmI2Qi1VPCRWOT9OXkklW21pLFhxSEIiMiNsaWknPmokUV5edF9DJj0iLSQ8TGk2LGpgPTtZTF46bmNoKSMoRTxRNGZQSmJOP2lBMEhjSTRUSFcpPE10Vj9kZFViMTtaT0khS1UhQkE4dEpCWldEa01ZWTBoPmNbZjpdXGplIjlsNmBPM21maUV0UWUtJlhHbFVSW0UycVkjUC4pPS9jWEFnKi9tPlJuSSpsYW9rS09mSy5YYDg+Z2cvSiVEL2s5YEU4biNnbG90XilBSCVoKWlYXkw/Kl5vWjUoRGk8Imw+a3VXSW1dY1hQaGZjX1UjLWs1UDVTNDonJk4wVEc1Q1lKLmFOZFwlN2ZBNjUrPV9CLWdgXm5wQ20mYG4lVU4hZ1wzRGBCVCRVZzooPC1cRl1baUYlZGlsMzB1L2E+O1ozNiVkckBVJ3AnL0RaQFZeVVYqLiMkOmpeI21MY1A9RCRFQlpVUTVvbmQtL1k+aitlNkpNTGgpIjYoXmA6VWs8OktwPzRGVyoqPj9QTCdPVVFibjpyZzAmYENxQT1tWz9DUUtISDtmNUwxPlBjZkIlLW1cL0dhPEJMXThBJV5ZQCxrcHUiPCRuKi9wWVtpajQ9M0ZtSFwvaVxNMSQkcWtZTGhYbyY/WjM+Kl5lK2p0Yk1GSGJMTmJpdSNrZF9pWD5pNUA2TSZLTDFBcGlzTydRZiUpWldrTDctQjpcTUhxLVg/cyM8RS1zYiEvO0woRGQwP3RGJyUrIUxvTzdgNWJDbEU+akVKMDloRmZBQWskblZKZmFbM001XW1SP1VPY05CVWU5MFVQRENLWCQ+MFBBLkQwajVvODcoNzUlTChlbzdcTWUrSGImYmxcSl84K1Vhcz1qXUNVNiRTRCY7PmEuLDpucFZQWzlKdTQiIStqIT42OFk2Y0EibSExcyk5bTwtNDA/Yz5AST5daWExU1VRM1tGZz5sLCtiallLUiJmbzZUVDhnRDNEJSMxOWVvOWFjPy4uPyVBX1M3RG9PbDI4VWNZcWExLyI8S145bickY1IjI1hlQVpmQTZBdGtwKDgpYDdHKTUwWkJbKiFGX2twallWQjJSbENVKUhUK25NUUQmJXB0bFIwZUxUM2dYbHFjbVdzWD5AUjJdJFYlUjpjPCY7dCI+OHI8UUM2LVc/R1ZrYyloVUc7V1M/USE2UScyLEFuOUR1TTs+ck5tMU5UX1U8cS5Zam80bT9cdDhmaidDSWVlP0VgY3QyKlBFIiElUUxkUE9rcWtIIy1KRSldb1poSGZlNV5gVlJZYzJUZ2BeN1RJXTRXZ2hEMEFRXnRgXUwlbERjazJwUzNXMnNJXzRcWm09a2JZPGdWJ1FrZ14vZ1lWJ1c9OVlebWAzayJhWyZKNUxsRWM0REtEIWkuQVYrPiImNz9ONThUanJVUURdQTE1WlQpQjFIPlkqVH4+ZW5kc3RyZWFtCmVuZG9iagoxNCAwIG9iago8PAovRmlsdGVyIFsgL0FTQ0lJODVEZWNvZGUgL0ZsYXRlRGVjb2RlIF0gL0xlbmd0aCAyMTM2Cj4+CnN0cmVhbQpHYXRVNGFgPy0qJkFAQltqc0UpVy4wKGJvQTotcWUwJ0E9Ol1HNWljUHFHRjM4PCssJl5wPl5pXi04Xz4iLExKOVBETW1hUkRkT3NsaEB1O1BgI3I3M1dDXU4rXlFiPU8yJ2ckMD5xPEdvJVRYPFokN2VVLzIiWUNqUVhkI0VBTy4lbzNDJ0Zuc2IialNgO2xnY000aFA/YmV1bChzVk1lLT1ZQUUyRGIyJzxsc0soQms9aWMmVDxwTkViVU5EKyY/bzFoS1o0dGQmSWpTXSJoQktdJSI1PFdsbydEbGczP0lWR1gpKTVKI1pBa04jZydnLF5tZS4kaEgxTEAyNEdDWGBgc0dTI2NUYDBGVzFQUkJHS0dOQUIuNUtFPUU6P1JuKi5SWkpfUFwtSDYtXFJcY0NYbDdnZychO1lCYmZIPFZjK15cYjNBOCwwTCRMJkpoaUhjblo/cWJKODFWZiRGZyxyRVsuQFJkR15qMVVtOV1fczdoJS1ePEsldG5VXzJHLm5oQHQoVks1UWwuQlJMZi5wRzsnKkcqYy11KCdePXA/OHRxaVx0Wk9yJD0jOG1APlg5MHBrMSQzQCUmU2c/WixKck4nPTAvXURVMllbOEZcQkdiOUJAa2cyMlA5L1JaMXE3JiJrZFhyWG1lcSlCZTpnOCI0RVNPIU4pREVDVXVtQSdONHNoXlc7X2hjYiwmVFNFXTdoREpaUURGL3ReK3BxZiNuIm0lNC5kSWhlWGQ6PTY6KEFWVzhPRHNYW0RKPikzXHVsXlYjNyxUNUYiLDtcXHIkbUUuTDA9RkYuM0w9VmosbV1sO0UvYE5BZUEpYyUoKFRXP2RoMyk8RF00ZTMnPzExQiFBRSI9WG1XNTJmLzpNSGYpOF5WayU8aUY+TEs7JF1sSGYxQzFXW1xVZXNLO1YjJCUubUZ1TG8tcCEzczxaSCsrNV9paV07NlMnSEMkRl9OJVA9Uj8sJ15gYlMzUypCX2skKj42WDgkQ3NqOy1jTCpmR3U/Myw5Ulk9cClHaCxaKkxASzVfVCFNPEBxbTglTWNccDtsQFlySDY5VTRcTVMiQVMhISFJaTkuczAmQkxrOzIvcVFWL2AhS1w1JnA8ZGotSClRQiNnU2tOJ3Mjb0ZBZzJHPlM1Q2o+cUhmKjNJN01hXV9CXj8vTiNwKV9GLTwraypGVSU6aDJZU04iV251YFdMTzdLRSI7KWklRCpzXTNuN29qVVEyNSJLRW8uZjpNP2onaVpcMCVuV0YhdVpdLShTJVBGYSskQjxmMTtucT02ckpsKlM/UDwwKEtbMUhPZXVbUmtLcjg1Njo8MVxTJVktVmRhcnE1QDRXMk4mLzgwWXNKRz88RXBWPWBgWT1XbFY/Yl9vLSgkJkVJKVlfVV4yK2RxLkMrRVJJSkdOPUNkVEk6PydBV145YykiWz4yJjpDWCJnTHJfSFNka1g7Tip0KzM5J0VdMGdqRzlmOGJwJmZWI0xNcjxoLCZQSVAsZjxuPWtJIWxuZ1QraHN0OyJwUXI6TGVgLj00cl9gJjtWJ1k1KlwwKk9CTVlcVi9pbzN0MmhobkoyWWFGM0w2Z2RHVExHZFheOTV0LjdzSnBJQjk7RGVRcDUoJF8wIWY+YFQyM2RJQF1ILWUoXmojIzNxbTwvNG44VyspLEpVW2MuVV9COihpZWdhS3M9PVUrWj1uIjViWmRnSjk4ZFhealo8cjpAYTZhKCxKM2hEbyE6V0UzbydSbWZFOnIqT2QxTktIbUdIbHBpXTc+MHNcVEREdG4sbzAiUjxbcWE+OHRzOVdhVj1Va0cqPWhvQjgzQ2VCQjs7RDVTWyRTJGYrLkw/T1FkJyxlSSQsKyFVIjFeK15mRl4mJlgpc0ojKjpTO3V0IVpyYk8oV01uNkBhWShUb2tEUjIrSk1GYikmUlowa1t1SFlhJzcrViEvZi9HOz9UKWNDIT8sWjpubCRnbCJUbjJsbmw4ZmhtUzg6TCdTOSVmYSsoLU45ZlxSWWU6K3FTIyxHZil0QkgqQlFeWSFNUXImUEMqXz4nREgvXjRyRUhBMiwnMF0hXkdQPyVHNEMuJzk1S1ZeXzZmKW4nK3UvYD1uZEslLV5JUkk5alReYjFUQUg9MlNxUXVocEdTXGUyLCUlJS1PKz9JLysxQE47SVVcVG0rbV5XNG9bcVtSZWZlSDoycUVLXmUzNFBPdVE6TkFUImZcLnVGXFtSWGNQbSEnYV0qZipsckA5OHM1UFxqPTFMSFM+VVROOzcoVWxcS0xrJCh0bG1FZGRsI0s3VjZkO1lrZz08U0EiK3QtWDgnNDFQSWE1JVZEL0xJQ0tFaCczUSxCWi8qRGA8bCRrJV8uW2RaNjxCTmYzNlpaKlRNWEVOWzFVW0I6MDZIYz5jSkEoXW86USRCZGk3TGhvWW1UKSxHOztFa0VtOm1kMldoVnNzTDdlM3QnWl51LFxyPVpGWDNmTjYzTVRMSjRGXERnQnMwWHJDMlZycGUkR2NZZ1teYmNBYltPPC88cTlyZEdzKXBAdDAjR2NkLDIpKzVfSj9wS1NeI148cCpIKjhpL2ZjTjNHRStnNztKLFk4dCU5QnAnLjpqRi0/bCphcmNwV2dfbTxMb0ppL0BgJ1tZPyRdV0huaVlkQ1I8Vk9damoiMFM4XEVrQkRFVU1FcltXLmNQNTA9NiRKZGBHSm11Pm9fVypETDZvOjlYSVMoSE5pXEYzYG9qRmdWXWs4OWliXDBNK251bk5XQ1k5cDdqNWBYa1NxUW1HbC1Bbzg3Rk9yZDZbNmJkPyVWNV5qaGw6YF5DZFInYFRKO3FWISkmaW8yJ1lEbm9sfj5lbmRzdHJlYW0KZW5kb2JqCjE1IDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoIDYxMAo+PgpzdHJlYW0KR2F0VS85b24hXidZTyNmanAlcTFNXW1hb1dqXzt0PC9ITDdSY0BeSkdRQiYwIUlwZDBeW0g2TGJ1MkpwPj8tTl1hRV1SajFcMy5PayFxN0Q7JCE7MGd0VzluJDFcXVE5aUMlRmd1WjRaOkhWXWpQOWFAKygnPSlxOEFsXVYwVExXPTppb3JPVCQzOXQ5RyQvQXBDaUAmZ2FTX00xJGZbLl0tbF0mTlZsPV0iRzxFWWYmPUNwMWNdQG5WX2JSZj81aTh0IyJFcSc8R2g2cSpWK1g1aiJvY3NYLlkzK0pzLXBkJ0hYLyRbdWcjIzpLMV4+UkgxU15CXmtoRSUtL2FLSXJXc0xhcVFvK1ZES28nNSQ0Z2FXMDYkZFtjMEE1ZkJjbm1zJ2B0Mlx1TmhyZlt1MC5IUXI+cmI7SVszVS9RSmMzK3U9IWV1VWJvU1ZxRypATlhBW29HXS8pZk1Tayg8MVszJCdcLzpnT3A4RDgiUGxzbFhuNG8pM2xlLkppO15aTkQiNE09ZSRrPF0jZy8vI2AnS3AnJlc6LUFjKWxaYmQ9N2taZTxWdW8+YzMqYjhRbyokM1pCUVpZKVIvbnJKK1hhcTI6Xi0ibU5SJl4rbyJqW2hoQ2NDVU00XjVJQmd1PklOVlZGXWI4aSNCSEUyLV43WiNAXSgycVdQWVxMI00yWD4+TGZiPTkyQWByOXJtcllcPls/dD5BT1QmLFU4XmU4XlxNJydCRjtvPT8vaUBlUU9hYFtGXiRBbDM7ZnUsSU0pXkV0JTtYVjh1OWo7OmAwcXAhUE1jSUs+YkcvXmp+PmVuZHN0cmVhbQplbmRvYmoKeHJlZgowIDE2CjAwMDAwMDAwMDAgNjU1MzUgZiAKMDAwMDAwMDA2MSAwMDAwMCBuIAowMDAwMDAwMTEyIDAwMDAwIG4gCjAwMDAwMDAyMTkgMDAwMDAgbiAKMDAwMDAwMDMzMSAwMDAwMCBuIAowMDAwMDAwNDUwIDAwMDAwIG4gCjAwMDAwMDA2NTUgMDAwMDAgbiAKMDAwMDAwMDg2MCAwMDAwMCBuIAowMDAwMDAxMDY1IDAwMDAwIG4gCjAwMDAwMDEyNzAgMDAwMDAgbiAKMDAwMDAwMTMzOSAwMDAwMCBuIAowMDAwMDAxNjgyIDAwMDAwIG4gCjAwMDAwMDE3NjAgMDAwMDAgbiAKMDAwMDAwNDI1NCAwMDAwMCBuIAowMDAwMDA2NTcyIDAwMDAwIG4gCjAwMDAwMDg4MDAgMDAwMDAgbiAKdHJhaWxlcgo8PAovSUQgCls8MjVhNTczZGQyMGQwMjMyN2Q5NTEzNzg0NGU3YzUyYjg+PDI1YTU3M2RkMjBkMDIzMjdkOTUxMzc4NDRlN2M1MmI4Pl0KJSBSZXBvcnRMYWIgZ2VuZXJhdGVkIFBERiBkb2N1bWVudCAtLSBkaWdlc3QgKG9wZW5zb3VyY2UpCgovSW5mbyAxMCAwIFIKL1Jvb3QgOSAwIFIKL1NpemUgMTYKPj4Kc3RhcnR4cmVmCjk1MDEKJSVFT0YK"
}
print(f"🛟 {len(PDF_SECOURS)} documents de secours embarqués. On n'en aura sans doute pas besoin.")

🛟 6 documents de secours embarqués. On n'en aura sans doute pas besoin.


## Jalon 1 · Les documents

Un assistant IA d'entreprise répond à partir de **documents**. Les nôtres, ce soir : les guides pratiques du **Club Odyssée**, une franchise de salles de sport avec cinq clubs en France (Paris, Lyon, Marseille, Toulouse, Lille) et ses conditions générales. Six PDF, qu'on va **télécharger** comme on le ferait avec les documents d'un client.

Pourquoi une franchise fictive ? Parce que l'IA ne peut rien en savoir. Si votre assistant répond juste tout à l'heure sur les horaires de Lyon, vous saurez que c'est grâce à VOTRE travail, pas à sa mémoire.

In [5]:
# Install required packages
get_ipython().system('pip install -U pypdf requests')

import io
import requests
from pypdf import PdfReader

# ⬇️ L'adresse publique où sont rangés les PDF (dataset Github) - using RAW URLs ⬇️
BASE_URL = "https://huggingface.co/datasets/Trixel3/zdoc/resolve/main/"
FICHIERS = ["Code-penal-Haitien97PAGES.pdf","USConstitution_French.pdf",
            "constitution-1958-vingt-cinquieme-revision.pdf","la_constitution_de_1987_amendee.pdf"
            ]

TEXTES = {}
success_count = 0

for f in FICHIERS:
    try:
        url = BASE_URL + f
        print(f"📥 Téléchargement de {f}...")
        r = requests.get(url, timeout=30)
        r.raise_for_status()

        # Check if we got valid PDF content
        content = r.content
        if not content.startswith(b'%PDF'):
            print(f"⚠️ {f} ne semble pas être un PDF valide (premiers octets: {content[:20]})")
            continue

        pdf_file = io.BytesIO(content)
        reader = PdfReader(pdf_file)
        pages = reader.pages
        text = "\n".join((p.extract_text() or "") for p in pages)
        TEXTES[f] = text
        print(f"✅ {f:42s} {len(pages)} page(s) · {len(text):5d} caractères")
        success_count += 1

    except requests.exceptions.RequestException as e:
        print(f"❌ Erreur de téléchargement pour {f}: {e}")
    except Exception as e:
        print(f"❌ Erreur de traitement pour {f}: {e}")

print(f"\n📚 {success_count}/{len(FICHIERS)} documents chargés avec succès.")

# Display first few characters of each document to verify
print("\n📄 Aperçu des documents:")
for f, text in TEXTES.items():
    preview = text[:200].replace('\n', ' ')
    print(f"{f[:30]}: {preview}...")

📥 Téléchargement de Code-penal-Haitien97PAGES.pdf...
✅ Code-penal-Haitien97PAGES.pdf              97 page(s) · 249490 caractères
📥 Téléchargement de USConstitution_French.pdf...
✅ USConstitution_French.pdf                  15 page(s) · 47595 caractères
📥 Téléchargement de constitution-1958-vingt-cinquieme-revision.pdf...
✅ constitution-1958-vingt-cinquieme-revision.pdf 40 page(s) · 87237 caractères
📥 Téléchargement de la_constitution_de_1987_amendee.pdf...
✅ la_constitution_de_1987_amendee.pdf        26 page(s) · 116686 caractères

📚 4/4 documents chargés avec succès.

📄 Aperçu des documents:
Code-penal-Haitien97PAGES.pdf: 1      Code pénal Haïtien.-   É d i t i o n  o r i g i n a l e  e n  v e r s i o n  i n t é g r a l e ,  V o t é  à  l a  C h a m b r e  d e s  C o m m u n e s ,  l e  2 9  j u i l l e t   1 8 3 5 ,  ...
USConstitution_French.pdf: CONSTITUTION DES ÉTATS-UNIS D'AMÉRIQUE  PRÉAMBULE  Nous, Peuple des États-Unis, en vue de former une Union plus parfaite, d'établir la  ju

### Le chunking : pourquoi on découpe

On ne va pas donner un document entier au modèle. Deux raisons :

1. **Sa fenêtre de contexte est finie** : on ne peut pas tout y mettre.
2. **Un passage court se retrouve mieux qu'un document entier.** Si vous cherchez les horaires de Lyon, vous voulez le paragraphe des horaires de Lyon, pas les quatre pages du guide.

Découper un texte en passages, c'est le **chunking**. Le nôtre est volontairement simple : des passages d'environ **500 caractères**, coupés de préférence en fin de phrase, avec un petit **chevauchement** pour ne pas perdre une information à cheval sur deux passages. En production, on fait plus sophistiqué. Ce soir, ça suffit largement.

Et pourquoi ne pas tout donner au modèle d'un coup ? Nos six guides font environ 25 pages, soit 15 000 tokens. Un modèle local s'y perd, c'est lent, et en entreprise on n'a pas 6 documents mais 6 000. On lui donne trois passages, les bons.

Une astuce de métier, gratuite : on **encode le titre du document avec le passage**. Ainsi le moteur sait que « ouvert de 6h30 à 22h30 » parle de Lyon, pas de Paris. En revanche on donne au modèle le passage **seul** : si on lui colle le titre dedans, il le recopie dans sa réponse. Détail minuscule, effet immédiat.

In [6]:
# Maintenant créer les passages
# Ré-exécuter la création des passages avec vérification
def decouper(texte, taille=500, chevauchement=80):
    """Découpe un texte en passages d'environ `taille` caractères."""
    if not texte or len(texte.strip()) == 0:
        return []

    texte = " ".join(texte.split())
    passages, debut = [], 0

    while debut < len(texte):
        fin = min(debut + taille, len(texte))
        if fin < len(texte):
            coupe = texte.rfind(". ", debut + taille // 2, fin)
            if coupe != -1:
                fin = coupe + 1
        passage = texte[debut:fin].strip()
        if passage:  # Ne garder que les passages non-vides
            passages.append(passage)
        if fin >= len(texte):
            break
        debut = max(fin - chevauchement, debut + 1)
    return passages

# Reconstruire DOCUMENTS avec vérification
DOCUMENTS = []
if 'TEXTES' in dir() and TEXTES:
    print(f"📚 Traitement de {len(TEXTES)} documents...")
    for f, t in TEXTES.items():
        nom = f.replace(".pdf", "").replace(".txt", "").replace("_", " ").title()
        passages = decouper(t)
        print(f"   - {nom}: {len(passages)} passages")
        for i, p in enumerate(passages, 1):
            DOCUMENTS.append({"titre": f"{nom} · passage {i}", "texte": p})
else:
    print("❌ TEXTES est vide ou inexistant.")
    print("💡 Création de documents de test...")

    # Créer des documents de test
    TEXTES = {
        "Constitution Haiti": "La Constitution de la République d'Haïti établit les principes fondamentaux du gouvernement. Elle garantit les droits des citoyens et définit le rôle des institutions. La Constitution est la loi suprême du pays.",
        "Code Pénal": "Le Code pénal haïtien définit les infractions et les peines. Il protège les personnes et les biens. Les sanctions incluent l'emprisonnement et les amendes."
    }

    for f, t in TEXTES.items():
        passages = decouper(t)
        for i, p in enumerate(passages, 1):
            DOCUMENTS.append({"titre": f"{f} · passage {i}", "texte": p})

# Affichage final avec vérification
print(f"\n✅ {len(DOCUMENTS)} passages créés au total.")

if DOCUMENTS:
    print("\n📝 Exemple de passage :")
    print(f"Titre: {DOCUMENTS[0]['titre']}")
    print(f"Texte: {DOCUMENTS[0]['texte'][:350]}...")
else:
    print("\n❌ Aucun passage n'a été créé.")
    print("Vérifiez que TEXTES contient du texte.")

📚 Traitement de 4 documents...
   - Code-Penal-Haitien97Pages: 699 passages
   - Usconstitution French: 139 passages
   - Constitution-1958-Vingt-Cinquieme-Revision: 247 passages
   - La Constitution De 1987 Amendee: 335 passages

✅ 1420 passages créés au total.

📝 Exemple de passage :
Titre: Code-Penal-Haitien97Pages · passage 1
Texte: 1 Code pénal Haïtien.- É d i t i o n o r i g i n a l e e n v e r s i o n i n t é g r a l e , V o t é à l a C h a m b r e d e s C o m m u n e s , l e 2 9 j u i l l e t 1 8 3 5 , a u S é n a t d e l a R é p u b l i q u e l e 1 0 a o u t 1 8 3 5 , P r o m u l g u é l e 1 1 a o û t 1 8 3 5 ....


In [8]:
DOCUMENTS[3]

{'titre': 'Code-Penal-Haitien97Pages · passage 4',
 'texte': '2. 58 43. 44. 59 45. 60 46. 62 47. 63 48. 65 49. 65 50. 66 51. 67 52. 53. 70 54. 71 55. 72 3 56. 74 57. 75 58. 77 59. 81 60. 82 61. 83 62. 85 63. et 63 bis 86 64. 65. 66. 67. 68. 91 69. 70. 92 71. 94 72. 95 73. 96 74. 97 75. 98 76. 100 77. 101 78. 102 79. 80. 108 81. 109 82. 110 83. 111 et 112 84. 113 85. 86. 87. 88. 89. 90. 91. 92. 93. 94. 125 95. 127 96. 129 97. 132 98. 133 99. 135 100. 138 101. 139 102. 140 103. 141 104. 142 105. 143 106. 144 107. 145 108. 146 109. 147 110. 148 111. 149 112.'}

## Jalon 2 · Le moteur qui comprend le sens

Comment retrouver le bon passage quand quelqu'un pose une question ? Pas avec des mots-clés : avec des **embeddings**.

Le principe : chaque passage est transformé en un vecteur, une liste de nombres qui capture son **sens**. La question subit le même sort. Il ne reste qu'à comparer les vecteurs : les plus proches sont les passages qui parlent de la même chose.

Le modèle qu'on utilise pour ça, vous l'avez déjà vu mardi : c'est **celui qui a cartographié les 2 117 offres d'emploi**.

In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np

# TROU 1 · le modèle de mardi
encodeur = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# On encode le TITRE avec le passage : le moteur saura ainsi a quoi se referencer.
# Mais le texte qu'on donnera au modèle, lui, restera propre.
textes = [f"{d['titre']} -{d['texte']}" for d in DOCUMENTS]
vecteurs = encodeur.encode(textes, normalize_embeddings=True)

print(f"✅ {len(vecteurs)} passages encodés.")
print(f"Chaque passage est devenu un vecteur de {vecteurs.shape[1]} nombres.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ 1420 passages encodés.
Chaque passage est devenu un vecteur de 384 nombres.


In [14]:
# TROU 2 · combien de passages on donne au modèle ?
def chercher(question, k=3):
    """Renvoie les k passages les plus proches de la question."""
    v_question = encodeur.encode(question, normalize_embeddings=True)
    similarites = vecteurs @ v_question
    indices = np.argsort(-similarites)[:k]
    return [(DOCUMENTS[i]["titre"], DOCUMENTS[i]["texte"], float(similarites[i]))
            for i in indices]

for titre, texte, score in chercher("quelle est votre preoccupation concernant les constitutions"):
    print(f"[{score:.2f}] {titre}")

print()
print("✅ Le moteur retrouve les bons passages!")

[0.66] Constitution-1958-Vingt-Cinquieme-Revision · passage 22
[0.65] La Constitution De 1987 Amendee · passage 324
[0.65] Constitution-1958-Vingt-Cinquieme-Revision · passage 80

✅ Le moteur retrouve les bons passages!


### La preuve que c'est du sens, pas des mots

Posons une question dont **aucun mot** n'apparaît dans le bon passage : « quand est ne Jesus». Le passage qui répond parle de « », jamais d'« arrêter ». Regardez.

In [18]:
for titre, texte, score in chercher("ou se trouve le club de l'Algerie?"):
    print(f"[{score:.2f}] {titre}")

[0.31] Constitution-1958-Vingt-Cinquieme-Revision · passage 231
[0.26] Code-Penal-Haitien97Pages · passage 107
[0.26] Constitution-1958-Vingt-Cinquieme-Revision · passage 4


## Jalon 3 · L'IA qui rédige... et d'abord, une expérience

Il nous manque la troisième brique : le modèle de langage qui va rédiger les réponses.

On va le charger, puis faire une expérience en deux temps:

1. On va lui poser une question **sans** lui donner le moindre document.
2. **Ensuite** on lui posera exactement la même, avec les documents. Et on affichera les deux réponses l'une sous l'autre.

Choisissez une question dont la réponse est **vérifiable** dans nos guides : un prix, un horaire, un équipement. Vous saurez ainsi, sans discussion possible, si la réponse est juste ou non.

In [19]:
from transformers import pipeline
import transformers

transformers.logging.set_verbosity_error()   # on masque les avertissements techniques

print("Chargement du modèle (1 à 3 minutes la première fois)...")

generateur = pipeline("text-generation", model= MODELE, device= DEVICE)

# On règle la génération une fois pour toutes
generateur.tokenizer.clean_up_tokenization_spaces = False
generateur.model.generation_config.max_new_tokens = 180
generateur.model.generation_config.do_sample = True
generateur.model.generation_config.temperature = None
generateur.model.generation_config.top_p = None
generateur.model.generation_config.top_k = None

print("✅ Modèle chargé !")



Chargement du modèle (1 à 3 minutes la première fois)...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Modèle chargé !


In [20]:
def demander_au_modele(messages):
    sortie = generateur(messages)
    return sortie[0]["generated_text"][-1]["content"]

In [21]:
# Dans toute vraie application, l'assistant a un RÔLE. Le nôtre :
ROLE = ("Tu es l'assistant des professionnels juridique, mini bibliotheque et consultant numerique. "
        "Tu réponds aux questions de façon précise, utile et directe.")

In [22]:
# votre question : sa réponse est dans nos guides, mais le modèle ne les a pas encore.
question_test = " article 2 de la constitution americaine"   # ex :

# On garde cette réponse de côté : on la comparera tout à l'heure.
reponse_sans_documents = demander_au_modele([
    {"role": "system", "content": ROLE},          # il est l'assistant
    {"role": "user", "content": question_test},   # ...mais il n'a aucun document.
])
print(reponse_sans_documents)

Article 2 de la Constitution américaine est le texte qui décrit les institutions du pays, leur structure et leurs fonctions. Voici une décomposition claire de cette partie :

1. Définition : Article 2 est un chapitre spécifique de la Constitution américaine, en particulier dans sa version actuelle.

2. Structure : Le texte comprend deux sections principales :
   - Section 1 : Établissement des institutions nationales.
   - Section 2 : Établissement des institutions locales.

3. Institutions nationales :
   - État : La Constitution national définit les institutions nationales (la Chambre des Representants, la Assemblée Nationale, etc.) et leurs fonctions spécifiques.
   - Élément légal : Définit les éléments législatifs de la Constitution nationale.

4. Institutions locales :
   - Élément légal : Définit les élus locaux (communes, districts, etc.) et leurs fonctions spécifiques.
   - État : Définit les élections locales.

5. Fonctionnement : Article 2 donne un cadre juridique pour le fo

### Que vient-il de se passer ?

Le Club Odyssée **n'existe pas**. Le modèle n'avait donc aucun moyen de connaître la réponse. Et pourtant il a produit quelque chose.

Regardez votre écran, puis le chat : **deux choses différentes viennent de se produire dans la salle.**

- Chez certains, il a **inventé** une réponse. Un prix, un horaire, avec assurance. C'est une **hallucination** : quand un modèle ne sait pas, il complète, parce que compléter est tout ce qu'il sait faire.
- Chez d'autres, il a **refusé** : « je n'ai pas cette information, contactez le club ».


Une seule cause aux deux pannes : **il n'a pas vos documents.**

Regardez bien la cellule suivante : **le rôle ne change pas**. Ce sont les documents et la règle qui s'ajoutent.

In [23]:
def repondre(question):
    """L'assistant complet : recherche + rédaction sous contrainte."""
    passages = chercher(question)# on recherches les K passages les plus pertinents vis-a-vis de la question
    contexte = "\n\n".join(f"### {t}\n{x}" for t, x, _ in passages)# on mets tous ces "passages" au sein d'une meme string, ca sera notre "contexte"

    # On Construit le Prompt Final : qu'est-ce qu'on lui interdit ? et que doit-il dire s'il ne sait pas ?
    messages = [
        {"role": "system", "content": ROLE +          # <- le MÊME rôle que tout à l'heure
            " Tu réponds uniquement à partir des documents fournis. "
            "Si la réponse n'y figure pas, réponds exactement : « Je ne sais pas, il faut consulter un professionnel du domaine »"},
        {"role": "user", "content": f"Documents :\n{contexte}\n\nQuestion : {question}"},
    ]

    reponse = demander_au_modele(messages)
    sources = ", ".join(t for t, _, _ in passages)
    return reponse, sources


# La MÊME question, cette fois avec les documents. On affiche les deux réponses l'une sous l'autre.
reponse_avec_documents, sources = repondre(question_test)

print("❓", question_test)
print()
print("❌ SANS les documents :")
print("   ", " ".join(reponse_sans_documents.split())[:400])
print()
print("✅ AVEC les documents :")
print("   ", " ".join(reponse_avec_documents.split()))
print("    📎 Sources :", sources)

❓  article 2 de la constitution americaine

❌ SANS les documents :
    Article 2 de la Constitution américaine est le texte qui décrit les institutions du pays, leur structure et leurs fonctions. Voici une décomposition claire de cette partie : 1. Définition : Article 2 est un chapitre spécifique de la Constitution américaine, en particulier dans sa version actuelle. 2. Structure : Le texte comprend deux sections principales : - Section 1 : Établissement des institut

✅ AVEC les documents :
    Article 2 de la Constitution américaine porte sur la constitutionale loi. Il est écrit comme suit : **Le Peuple Haïtien proclame la présente Constitution:** Pour garantir ses droits inalienables et imprescriptibles à la vie, à la liberté et à la poursuite du bonheur; conformément à son Acte d'indépendance de 1804 et à la Déclaration universelle des droits de l'homme et du citoyen.
    📎 Sources : La Constitution De 1987 Amendee · passage 324, La Constitution De 1987 Amendee · passage 1, Usconsti

In [24]:
ma_question = "le vol a mains armees"
passages = chercher(ma_question)# on recherches les K passages les plus pertinents vis-a-vis de la question
contexte = "\n\n".join(f"### {t}\n{x}" for t, x, _ in passages)# on mets tous ces "passages" au sein d'une meme string, ca sera notre "contexte"

contexte

"### Code-Penal-Haitien97Pages · passage 84\ndix ans, sous la surveillance spéciale de la haute police de l'État. 19 Art. Fr. 101 .- 77.- Sont compris dans le mot armes, toutes machines, tous instruments ou ustensiles tranchants ou contondants. Les couteaux et ciseaux de poche, les cannes simples ne seront réputés armes qu'autant qu'il en aura été fait usage pour tuer, blesser, ou frapper. III DISPOSITION COMMUNE AUX DEUX PARAGRAPHES DE LA PRÉSENTE SECTION Art.\n\n### Code-Penal-Haitien97Pages · passage 339\ns, instituteurs ou institutrices de l'enfant. II. ENLÈVEMENT DE MINEURS Art. Fr. 354 .- 300.- Quiconque aura, par fraude ou violence, enlevé ou fait enlever des mineurs, ou les aura entraînés, détournés ou déplacés, ou les aura fait entraîner, déto urner ou déplacer des lieux où ils étaient mis par ceux à l'autorité ou à la direction desquels ils étaient soumis ou confiés, subira la peine de la réclusion. Art. Fr.\n\n### La Constitution De 1987 Amendee · passage 310\nce. Article 26

In [25]:
questions = ["article 2 de la constitution haitienne ?",
             "Quelle est la devise nationale d'Haïti ?",
             "Quelles sont les langues officielles de la République ?",
             "Quelle est l'unité monétaire nationale ?",
             "Quel est l'hymne national ?"
             "L'effigie d'une personne vivante peut-elle figurer sur la monnaie ou les bâtiments publics? "
             ]

for q in questions:
    r, s = repondre(q)
    print(f"❓ {q}")
    print(f"💬 {r}")
    print(f"📎 {s}")
    print()

print("🎉 Levez la main dans le chat : votre assistant vient de répondre !")

❓ article 2 de la constitution haitienne ?
💬 Article 2 de la Constitution Haitienne est déclarant que "La société de la République d'Haiti est un État libre et indépendant." Cette affirmation reflète la nature fondamentale de la République d'Haiti, qui est une société en libre accord avec elle-même. Elle garantit également l'égalité entre tous les citoyens, respecte les droits humains et moraux, et se concentre sur l'amélioration constante de la vie de tous.
📎 La Constitution De 1987 Amendee · passage 2, La Constitution De 1987 Amendee · passage 318, La Constitution De 1987 Amendee · passage 15

❓ Quelle est la devise nationale d'Haïti ?
💬 La devise nationale d'Haïti est la monnaie Haïtienne. Elle est appelée "la Réserve Haïtienne de Bourses" (RHB) et elle est utilisée depuis 1975 pour tous les actifs financiers et commerciaux dans le pays.
📎 Code-Penal-Haitien97Pages · passage 117, Code-Penal-Haitien97Pages · passage 111, Code-Penal-Haitien97Pages · passage 62

❓ Quelles sont les lang

Notez la dernière question : le Bitcoin n'est pas dans les documents, et l'assistant **le dit** au lieu d'inventer. C'est exactement la différence entre utiliser l'IA et la maîtriser.

Et remarquez ce qui a changé entre les deux réponses de tout à l'heure : **pas le modèle**, il est identique. **Pas le rôle**, il est identique. Ce sont les trois passages et une règle de huit mots. C'est tout le RAG.

## Jalon 4 · LA MISE EN LIGNE

Votre assistant fonctionne. Mais il vit dans ce notebook, sur cette page. Personne d'autre que vous ne peut s'en servir.

On va lui donner deux choses : un **visage** (une vraie page web) et une **adresse publique**. Un seul mot crée un tunnel : votre application, qui tourne ici, devient accessible depuis n'importe où dans le monde.

In [26]:
import gradio as gr

def assistant_web(question):
    if not question.strip():
        return "Posez moi une question de droit!"
    reponse, sources = repondre(question)
    return f"{reponse}\n\n📎 Sources : {sources}"

demo = gr.Interface(
    fn=assistant_web,
    inputs=gr.Textbox(label="Votre question",
                      placeholder="Ex : quel est le coup penal du viol en Haiti ?"),
    outputs=gr.Textbox(label="Réponse de l'assistant"),
    title="Mon Super Assistant de droit",
    description="Il répond aux questions de droit constitutionnel "
                "(Haiti,France,USA) Construit en direct avec Machine Learnia.",
)

# TROU 5 · un seul mot pour passer de votre écran au monde
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://08d13be9961b837314.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 📱 Le moment

Dans la sortie ci-dessus, cherchez la ligne :

> `Running on public URL: https://xxxxxxxx.gradio.live`

Maintenant, dans l'ordre :

1. **Ouvrez ce lien dans un autre navigateur, ou sur votre téléphone.**
2. **Posez une question** à votre assistant.
3. **Envoyez le lien à quelqu'un. Maintenant.** Votre conjoint, un ami. Avec le message de votre choix. Le nôtre serait : « regarde ce que je viens de construire ».

Puis revenez nous dire dans le chat ce que ça fait. 😊

**À savoir** : ce lien est temporaire, il vivra **quelques heures** puis s'éteindra. Donc si vous voulez le partager avec d'autres personnes (par exemple sur linkedin, NE PARTAGER PAS CET URL... demain il sera désactivé)

## Le bilan

Ce soir, vous avez construit les quatre briques de tout assistant IA d'entreprise :

1. **Des documents** allés chercher sur Internet, puis **découpés en passages** (le chunking).
2. **Un moteur de recherche sémantique**, qui comprend le sens et pas seulement les mots.
3. **Un modèle qui rédige sous contrainte** : il répond à partir de vos documents, cite ses sources, et dit « je ne sais pas » plutôt que d'inventer.
4. **Une mise en ligne**, à une adresse publique.

Ce qui est temporaire, c'est le lien. Ce qui est à vous, c'est le savoir-faire, et ce notebook.

**Pour les bâtisseurs du Cahier de Vacances** : votre application du Projet 7 mérite le même traitement. La méthode change un peu (Streamlit a son propre service de mise en ligne, gratuit lui aussi), le principe est identique.

À samedi ! ✌️

**Guillaume - Machine Learnia**

```

# LES RÉPONSES DU TP :

1. paraphrase-multilingual-MiniLM-L12-v2

2. [f"{d['titre']} - {d['texte']}" for d in DOCUMENTS]

3. k=3

4.
print("Chargement du modèle (1 à 3 minutes la première fois)...")
generateur = pipeline("text-generation", model=MODELE, device=DEVICE)

# On règle la génération une fois pour toutes
generateur.tokenizer.clean_up_tokenization_spaces = False
generateur.model.generation_config.max_new_tokens = 180
generateur.model.generation_config.do_sample = False
generateur.model.generation_config.temperature = None
generateur.model.generation_config.top_p = None
generateur.model.generation_config.top_k = None



5.
passages = chercher(ma_question) # on recherches les K passages les plus pertinents vis-a-vis de la question
contexte = "\n\n".join(f"### {t}\n{x}" for t, x, _ in passages)


6. UNIQUEMENT

```